In [2]:
!pip install -q datasets

In [3]:
from datasets import load_dataset

dataset = load_dataset("tasksource/ruletaker")

print(dataset)

README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/train-00000-of-00001-52adaa842dd7ed(…): reconstructing file:   0%|          |  0.00B / 28.6MB            

data/train-00000-of-00001-52adaa842dd7ed(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-402c217dbcd0259(…): reconstructing file:   0%|          |  0.00B / 3.72MB            

data/test-00000-of-00001-402c217dbcd0259(…): downloading bytes:           |  0.00B            

data/dev-00000-of-00001-480b21836eb3db96(…): reconstructing file:   0%|          |  0.00B / 1.85MB            

data/dev-00000-of-00001-480b21836eb3db96(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/480152 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/151911 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/75872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 480152
    })
    test: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 151911
    })
    dev: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 75872
    })
})


In [4]:
print(dataset)

split_name = list(dataset.keys())[0]
print("Selected split:", split_name)

print(dataset[split_name][0])

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 480152
    })
    test: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 151911
    })
    dev: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 75872
    })
})
Selected split: train
{'context': 'Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.', 'question': 'Bob is kind.', 'label': 'entailment', 'config': 'depth-1'}


In [5]:

# Check which labels are present
print("Labels:", dataset["train"].unique("label"))

# Check how many examples belong to each label
from collections import Counter

label_counts = Counter(dataset["train"]["label"])
print("Label counts:", label_counts)

# Inspect 3 development examples
for i in range(3):
    print("\nExample", i + 1)
    print("Context:", dataset["dev"][i]["context"])
    print("Question:", dataset["dev"][i]["question"])
    print("Label:", dataset["dev"][i]["label"])


Labels: ['entailment', 'not entailment']
Label counts: Counter({'entailment': 241543, 'not entailment': 238609})

Example 1
Context: The cat chases the cow. The cat is big. The cat is red. The cat is round. The cat is young. The cat needs the tiger. The cow chases the lion. The cow likes the tiger. The cow needs the cat. The lion chases the tiger. The lion is young. The tiger needs the lion. If someone likes the cat then they are young. If the cow chases the lion then the lion needs the tiger. If someone chases the cat then they are cold. If someone is young then they chase the lion. If someone likes the cow and the cow chases the lion then the cow chases the tiger. If someone likes the lion and they chase the lion then the lion is cold. If someone likes the tiger and the tiger needs the cow then they chase the cow. If someone needs the cat and the cat likes the tiger then they are young.
Question: The cow likes the tiger.
Label: entailment

Example 2
Context: The cat chases the cow. T

In [6]:

def forward_chaining(facts, rules):
    """
    facts: set of known facts
    rules: list of (conditions, conclusion)
    """
    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:
            # Apply a rule if all its conditions are known
            if all(condition in known_facts for condition in conditions):
                if conclusion not in known_facts:
                    known_facts.add(conclusion)
                    changed = True

    return known_facts


# Example facts
facts = {
    ("young", "Bob"),
    ("kind", "Bob")
}

# Example rules
rules = [
    ([("young", "Bob")], ("smart", "Bob")),
    ([("smart", "Bob"), ("kind", "Bob")],
     ("successful", "Bob"))
]

# Run the reasoning engine
result = forward_chaining(facts, rules)

print("Derived facts:")
for fact in sorted(result):
    print(fact)

# Check the query
query = ("successful", "Bob")
print("\nQuery:", query)
print("Prediction:", query in result)


Derived facts:
('kind', 'Bob')
('smart', 'Bob')
('successful', 'Bob')
('young', 'Bob')

Query: ('successful', 'Bob')
Prediction: True


In [7]:

def forward_chaining(facts, rules):
    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:
            # Find substitutions satisfying every condition
            substitutions = [{}]

            for predicate, arguments in conditions:
                new_substitutions = []

                for substitution in substitutions:
                    for fact_predicate, fact_arguments in known_facts:
                        if predicate != fact_predicate:
                            continue
                        if len(arguments) != len(fact_arguments):
                            continue

                        updated = substitution.copy()
                        match = True

                        for pattern, value in zip(arguments, fact_arguments):
                            if pattern.startswith("?"):
                                if pattern in updated and updated[pattern] != value:
                                    match = False
                                    break
                                updated[pattern] = value
                            elif pattern != value:
                                match = False
                                break

                        if match:
                            new_substitutions.append(updated)

                substitutions = new_substitutions

            # Apply each valid substitution to the conclusion
            for substitution in substitutions:
                derived = (
                    conclusion[0],
                    tuple(substitution.get(arg, arg) for arg in conclusion[1])
                )

                # Facts use the same (predicate, arguments) structure
                if derived not in known_facts:
                    known_facts.add(derived)
                    changed = True

    return known_facts


# All facts use the format: (predicate, tuple(arguments))
facts = {
    ("young", ("Bob",)),
    ("kind", ("Bob",)),
    ("young", ("Alice",)),
}

# Variables begin with ?
rules = [
    ([("young", ("?x",))], ("smart", ("?x",))),
    ([("smart", ("?x",)), ("kind", ("?x",))],
     ("successful", ("?x",)))
]

result = forward_chaining(facts, rules)

print("Derived facts:")
for fact in sorted(result):
    print(fact)

print("\nBob successful:",
      ("successful", ("Bob",)) in result)

print("Alice smart:",
      ("smart", ("Alice",)) in result)

print("Alice successful:",
      ("successful", ("Alice",)) in result)


Derived facts:
('kind', ('Bob',))
('smart', ('Alice',))
('smart', ('Bob',))
('successful', ('Bob',))
('young', ('Alice',))
('young', ('Bob',))

Bob successful: True
Alice smart: True
Alice successful: False


In [8]:

import re

def parse_fact(sentence):
    sentence = sentence.strip().rstrip(".")

    # Example: "Bob is young"
    match = re.fullmatch(
        r"([A-Za-z]+) is ([A-Za-z]+)",
        sentence
    )

    if match:
        entity, predicate = match.groups()
        return (predicate.lower(), (entity,))

    # Example: "Bob chases the cow"
    match = re.fullmatch(
        r"([A-Za-z]+) ([A-Za-z]+) the ([A-Za-z]+)",
        sentence
    )

    if match:
        subject, predicate, obj = match.groups()
        return (predicate.lower(), (subject, obj))

    return None


# Try the parser
sentences = [
    "Bob is young.",
    "Bob is kind.",
    "The cat chases the cow."
]

for sentence in sentences:
    print(sentence, "->", parse_fact(sentence))


Bob is young. -> ('young', ('Bob',))
Bob is kind. -> ('kind', ('Bob',))
The cat chases the cow. -> None


In [9]:

import re

def parse_fact(sentence):
    sentence = sentence.strip().rstrip(".")

    # Pattern 1: Bob is young.
    match = re.fullmatch(
        r"([A-Za-z]+) is ([A-Za-z]+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        entity, predicate = match.groups()
        return (predicate.lower(), (entity,))

    # Pattern 2: The cat chases the cow.
    # Also supports: Bob chases the cow.
    match = re.fullmatch(
        r"(?:The )?([A-Za-z]+) ([A-Za-z]+) (?:the )?([A-Za-z]+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        subject, predicate, obj = match.groups()
        return (predicate.lower(), (subject, obj))

    return None


# Test the parser
sentences = [
    "Bob is young.",
    "Bob is kind.",
    "The cat chases the cow.",
    "The cow likes the tiger.",
    "The lion needs the tiger."
]

for sentence in sentences:
    print(sentence, "->", parse_fact(sentence))


Bob is young. -> ('young', ('Bob',))
Bob is kind. -> ('kind', ('Bob',))
The cat chases the cow. -> ('chases', ('cat', 'cow'))
The cow likes the tiger. -> ('likes', ('cow', 'tiger'))
The lion needs the tiger. -> ('needs', ('lion', 'tiger'))


In [10]:

import re

def parse_fact(sentence):
    sentence = sentence.strip().rstrip(".")

    # Negative property: Anne is not young.
    match = re.fullmatch(
        r"([A-Za-z]+) is not ([A-Za-z]+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        entity, predicate = match.groups()
        return ("not_" + predicate.lower(), (entity,))

    # Positive property: Bob is young.
    match = re.fullmatch(
        r"([A-Za-z]+) is ([A-Za-z]+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        entity, predicate = match.groups()
        return (predicate.lower(), (entity,))

    # Relationship: The cat chases the cow.
    match = re.fullmatch(
        r"(?:The )?([A-Za-z]+) ([A-Za-z]+) (?:the )?([A-Za-z]+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        subject, predicate, obj = match.groups()
        return (predicate.lower(), (subject, obj))

    return None


# Test positive and negative facts
sentences = [
    "Bob is young.",
    "Anne is not young.",
    "The cat chases the cow."
]

for sentence in sentences:
    print(sentence, "->", parse_fact(sentence))


Bob is young. -> ('young', ('Bob',))
Anne is not young. -> ('not_young', ('Anne',))
The cat chases the cow. -> ('chases', ('cat', 'cow'))


In [11]:

import re

def parse_rule(sentence):
    sentence = sentence.strip().rstrip(".")

    # Pattern 1:
    # If someone is young then they are smart.
    match = re.fullmatch(
        r"If someone is (\w+) then they are (\w+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        condition, conclusion = match.groups()

        return (
            [(condition.lower(), ("?x",))],
            (conclusion.lower(), ("?x",))
        )

    # Pattern 2:
    # If someone chases the cat then they are cold.
    match = re.fullmatch(
        r"If someone (\w+) the (\w+) then they are (\w+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        relation, obj, conclusion = match.groups()

        return (
            [(relation.lower(), ("?x", obj.lower()))],
            (conclusion.lower(), ("?x",))
        )

    return None


# Test the rule parser
rules_text = [
    "If someone is young then they are smart.",
    "If someone chases the cat then they are cold."
]

for rule in rules_text:
    print(rule)
    print("Parsed:", parse_rule(rule))
    print()


If someone is young then they are smart.
Parsed: ([('young', ('?x',))], ('smart', ('?x',)))

If someone chases the cat then they are cold.
Parsed: ([('chases', ('?x', 'cat'))], ('cold', ('?x',)))



In [12]:

import re

def parse_rule(sentence):
    sentence = sentence.strip().rstrip(".")

    # Rule with two conditions:
    # If someone likes the cow and the cow chases the lion
    # then the cow chases the tiger.

    match = re.fullmatch(
        r"If someone likes the (\w+) and the (\w+) "
        r"chases the (\w+) then the (\w+) chases the (\w+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        obj1, subject2, obj2, subject3, obj3 = match.groups()

        return (
            [
                ("likes", ("?x", obj1.lower())),
                ("chases", (subject2.lower(), obj2.lower()))
            ],
            ("chases", (subject3.lower(), obj3.lower()))
        )

    # Rule with one condition:
    # If someone is young then they are smart.
    match = re.fullmatch(
        r"If someone is (\w+) then they are (\w+)",
        sentence,
        re.IGNORECASE
    )

    if match:
        condition, conclusion = match.groups()

        return (
            [(condition.lower(), ("?x",))],
            (conclusion.lower(), ("?x",))
        )

    return None


# Test
rule = (
    "If someone likes the cow and the cow chases the lion "
    "then the cow chases the tiger."
)

print("Original rule:", rule)
print("Parsed rule:", parse_rule(rule))


Original rule: If someone likes the cow and the cow chases the lion then the cow chases the tiger.
Parsed rule: ([('likes', ('?x', 'cow')), ('chases', ('cow', 'lion'))], ('chases', ('cow', 'tiger')))


In [118]:



def match_condition(condition, facts, substitution):
    predicate, arguments = condition
    matches = []

    # Handle negative conditions
    if predicate.startswith("not_"):
        positive_predicate = predicate[4:]

        # First, try matching explicitly stated negative facts
        for fact_predicate, fact_arguments in facts:
            if fact_predicate != predicate:
                continue

            if len(arguments) != len(fact_arguments):
                continue

            updated = substitution.copy()
            valid = True

            for pattern, value in zip(arguments, fact_arguments):
                if pattern.startswith("?"):
                    if pattern in updated and updated[pattern] != value:
                        valid = False
                        break
                    updated[pattern] = value
                elif pattern != value:
                    valid = False
                    break

            if valid:
                matches.append(updated)

        if matches:
            return matches

        # Otherwise, use negation as failure
        grounded_arguments = []

        for arg in arguments:
            if arg.startswith("?"):
                if arg not in substitution:
                    return []
                grounded_arguments.append(substitution[arg])
            else:
                grounded_arguments.append(arg)

        positive_fact = (
            positive_predicate,
            tuple(grounded_arguments)
        )

        if positive_fact not in facts:
            return [substitution.copy()]

        return []

    # Handle positive conditions
    for fact_predicate, fact_arguments in facts:
        if predicate != fact_predicate:
            continue

        if len(arguments) != len(fact_arguments):
            continue

        updated = substitution.copy()
        valid = True

        for pattern, value in zip(arguments, fact_arguments):
            if pattern.startswith("?"):
                if pattern in updated and updated[pattern] != value:
                    valid = False
                    break
                updated[pattern] = value
            elif pattern != value:
                valid = False
                break

        if valid:
            matches.append(updated)

    return matches



def forward_chaining(facts, rules):

    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:

            substitutions = [{}]

            # Match all conditions of the rule
            for condition in conditions:

                new_substitutions = []

                for substitution in substitutions:
                    matches = match_condition(
                        condition,
                        known_facts,
                        substitution
                    )
                    new_substitutions.extend(matches)

                substitutions = new_substitutions

                if not substitutions:
                    break

            # Derive the conclusion
            for substitution in substitutions:

                predicate, arguments = conclusion

                derived_arguments = tuple(
                    substitution[arg] if arg.startswith("?") else arg
                    for arg in arguments
                )

                derived_fact = (predicate, derived_arguments)

                if derived_fact not in known_facts:
                    known_facts.add(derived_fact)
                    changed = True

    return known_facts


# ---------------------------------
# TEST THE COMPLETE REASONING ENGINE
# ---------------------------------

facts = {
    ("young", ("Bob",)),
    ("kind", ("Bob",)),
    ("young", ("Alice",)),
    ("likes", ("dog", "cow")),
    ("chases", ("cow", "lion"))
}

rules = [
    # If someone is young, they are smart.
    (
        [("young", ("?x",))],
        ("smart", ("?x",))
    ),

    # If someone is smart and kind, they are successful.
    (
        [("smart", ("?x",)), ("kind", ("?x",))],
        ("successful", ("?x",))
    ),

    # If someone likes the cow and the cow chases the lion,
    # then the cow chases the tiger.
    (
        [
            ("likes", ("?x", "cow")),
            ("chases", ("cow", "lion"))
        ],
        ("chases", ("cow", "tiger"))
    )
]

result = forward_chaining(facts, rules)

print("ALL DERIVED FACTS:\n")

for fact in sorted(result):
    print(fact)

# Check queries
queries = [
    ("successful", ("Bob",)),
    ("smart", ("Alice",)),
    ("chases", ("cow", "tiger"))
]

print("\nQUERY RESULTS:\n")

for query in queries:
    print(query, "->", query in result)


ALL DERIVED FACTS:

('chases', ('cow', 'lion'))
('chases', ('cow', 'tiger'))
('kind', ('Bob',))
('likes', ('dog', 'cow'))
('smart', ('Alice',))
('smart', ('Bob',))
('successful', ('Bob',))
('young', ('Alice',))
('young', ('Bob',))

QUERY RESULTS:

('successful', ('Bob',)) -> True
('smart', ('Alice',)) -> True
('chases', ('cow', 'tiger')) -> True


In [119]:
context = """
Charlie is blue.
If someone is blue and not rough then they are big.
"""

facts, rules, unsupported = parse_context(context)
derived = forward_chaining(facts, rules)

print("Derived facts:", derived)
print("Charlie is big:",
      ("big", ("charlie",)) in derived)

Derived facts: {('big', ('charlie',)), ('blue', ('charlie',))}
Charlie is big: True


In [120]:
context = """
Bob is not smart.
If something is not smart then it is nice.
"""

facts, rules, unsupported = parse_context(context)
derived = forward_chaining(facts, rules)

print("Bob is nice:",
      ("nice", ("bob",)) in derived)

Bob is nice: True


In [14]:

def match_condition(condition, facts, substitution):
    """Find facts that satisfy one rule condition."""

    predicate, arguments = condition
    matches = []

    for fact_predicate, fact_arguments in facts:

        if predicate != fact_predicate:
            continue

        if len(arguments) != len(fact_arguments):
            continue

        updated = substitution.copy()
        valid = True

        for pattern, value in zip(arguments, fact_arguments):

            if pattern.startswith("?"):
                if pattern in updated:
                    if updated[pattern] != value:
                        valid = False
                        break
                else:
                    updated[pattern] = value

            elif pattern != value:
                valid = False
                break

        if valid:
            matches.append(updated)

    return matches


def forward_chaining(facts, rules):

    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:

            substitutions = [{}]

            # Match all conditions of the rule
            for condition in conditions:

                new_substitutions = []

                for substitution in substitutions:
                    matches = match_condition(
                        condition,
                        known_facts,
                        substitution
                    )
                    new_substitutions.extend(matches)

                substitutions = new_substitutions

                if not substitutions:
                    break

            # Derive the conclusion
            for substitution in substitutions:

                predicate, arguments = conclusion

                derived_arguments = tuple(
                    substitution[arg] if arg.startswith("?") else arg
                    for arg in arguments
                )

                derived_fact = (predicate, derived_arguments)

                if derived_fact not in known_facts:
                    known_facts.add(derived_fact)
                    changed = True

    return known_facts


# ---------------------------------
# TEST THE COMPLETE REASONING ENGINE
# ---------------------------------

facts = {
    ("young", ("Bob",)),
    ("kind", ("Bob",)),
    ("young", ("Alice",)),
    ("likes", ("dog", "cow")),
    ("chases", ("cow", "lion"))
}

rules = [
    # If someone is young, they are smart.
    (
        [("young", ("?x",))],
        ("smart", ("?x",))
    ),

    # If someone is smart and kind, they are successful.
    (
        [("smart", ("?x",)), ("kind", ("?x",))],
        ("successful", ("?x",))
    ),

    # If someone likes the cow and the cow chases the lion,
    # then the cow chases the tiger.
    (
        [
            ("likes", ("?x", "cow")),
            ("chases", ("cow", "lion"))
        ],
        ("chases", ("cow", "tiger"))
    )
]

result = forward_chaining(facts, rules)

print("ALL DERIVED FACTS:\n")

for fact in sorted(result):
    print(fact)

# Check queries
queries = [
    ("successful", ("Bob",)),
    ("smart", ("Alice",)),
    ("chases", ("cow", "tiger"))
]

print("\nQUERY RESULTS:\n")

for query in queries:
    print(query, "->", query in result)


ALL DERIVED FACTS:

('chases', ('cow', 'lion'))
('chases', ('cow', 'tiger'))
('kind', ('Bob',))
('likes', ('dog', 'cow'))
('smart', ('Alice',))
('smart', ('Bob',))
('successful', ('Bob',))
('young', ('Alice',))
('young', ('Bob',))

QUERY RESULTS:

('successful', ('Bob',)) -> True
('smart', ('Alice',)) -> True
('chases', ('cow', 'tiger')) -> True


In [15]:

# Inspect examples containing negative questions
count = 0

for example in dataset["train"]:
    question = example["question"]

    if "not" in question.lower():
        print("\nContext:", example["context"])
        print("Question:", question)
        print("Label:", example["label"])
        print("Config:", example["config"])

        count += 1

    if count == 10:
        break



Context: Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.
Question: Dave is not smart.
Label: not entailment
Config: depth-1

Context: Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.
Question: Bob is not smart.
Label: entailment
Config: depth-1

Context: Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.
Question: Dave is not quiet.
Label: entailment
Config: depth-1

Context: Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are

In [16]:

# Inspect a few actual RuleTaker contexts

for i in range(5):
    example = dataset["train"][i]

    print("\n" + "="*60)
    print("CONTEXT:")
    print(example["context"])

    print("\nQUESTION:")
    print(example["question"])

    print("\nLABEL:")
    print(example["label"])



CONTEXT:
Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.

QUESTION:
Bob is kind.

LABEL:
entailment

CONTEXT:
Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.

QUESTION:
Dave is not smart.

LABEL:
not entailment

CONTEXT:
Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.

QUESTION:
Bob is not smart.

LABEL:
entailment

CONTEXT:
Anne is quiet. Anne is not young. Bob is kind. Bob is young. Dave is rough. Dave is round. Dave is smart. Dave is not young. Fiona is quiet. Fiona is not round. Kind, young things are not smart.

QUESTION:
Bob is smart.

LABEL:
not

In [17]:
def match_condition(condition, facts, substitution):
    predicate, arguments = condition
    matches = []

    # Handle negated conditions using negation as failure
    if predicate.startswith("not_"):
        positive_predicate = predicate[4:]

        # Check whether the corresponding positive fact exists
        for fact_predicate, fact_arguments in facts:
            if fact_predicate != positive_predicate:
                continue

            if len(arguments) != len(fact_arguments):
                continue

            updated = substitution.copy()
            valid = True

            for pattern, value in zip(arguments, fact_arguments):
                if pattern.startswith("?"):
                    if pattern in updated and updated[pattern] != value:
                        valid = False
                        break
                    updated[pattern] = value
                elif pattern != value:
                    valid = False
                    break

            if valid:
                # Positive fact exists, so negative condition fails
                return []

        # No matching positive fact: negative condition succeeds
        return [substitution.copy()]

    # Handle normal positive conditions
    for fact_predicate, fact_arguments in facts:
        if predicate != fact_predicate:
            continue

        if len(arguments) != len(fact_arguments):
            continue

        updated = substitution.copy()
        valid = True

        for pattern, value in zip(arguments, fact_arguments):
            if pattern.startswith("?"):
                if pattern in updated and updated[pattern] != value:
                    valid = False
                    break
                updated[pattern] = value
            elif pattern != value:
                valid = False
                break

        if valid:
            matches.append(updated)

    return matches


def forward_chaining(facts, rules):
    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:
            substitutions = [{}]

            for condition in conditions:
                new_substitutions = []

                for substitution in substitutions:
                    new_substitutions.extend(
                        match_condition(
                            condition,
                            known_facts,
                            substitution
                        )
                    )

                substitutions = new_substitutions

                if not substitutions:
                    break

            for substitution in substitutions:
                predicate, arguments = conclusion

                derived_arguments = tuple(
                    substitution[arg] if arg.startswith("?") else arg
                    for arg in arguments
                )

                derived = (predicate, derived_arguments)

                if derived not in known_facts:
                    known_facts.add(derived)
                    changed = True

    return known_facts


# Facts from our example
facts = {
    ("kind", ("Bob",)),
    ("young", ("Bob",)),
    ("smart", ("Dave",)),
    ("not_young", ("Anne",))
}

# Rules
rules = [
    (
        [
            ("kind", ("?x",)),
            ("young", ("?x",))
        ],
        ("not_smart", ("?x",))
    )
]

# Run reasoning
result = forward_chaining(facts, rules)

print("Derived facts:\n")

for fact in sorted(result):
    print(fact)

# Test queries
queries = [
    ("kind", ("Bob",)),
    ("not_smart", ("Bob",)),
    ("smart", ("Bob",)),
    ("not_smart", ("Dave",))
]

print("\nQuery results:\n")

for query in queries:
    print(query, "->", query in result)


Derived facts:

('kind', ('Bob',))
('not_smart', ('Bob',))
('not_young', ('Anne',))
('smart', ('Dave',))
('young', ('Bob',))

Query results:

('kind', ('Bob',)) -> True
('not_smart', ('Bob',)) -> True
('smart', ('Bob',)) -> False
('not_smart', ('Dave',)) -> False


In [18]:

# Inspect examples involving positive and negative queries

for i in range(10):
    example = dataset["train"][i]

    print("\nQuestion:", example["question"])
    print("Label:", example["label"])
    print("Config:", example["config"])



Question: Bob is kind.
Label: entailment
Config: depth-1

Question: Dave is not smart.
Label: not entailment
Config: depth-1

Question: Bob is not smart.
Label: entailment
Config: depth-1

Question: Bob is smart.
Label: not entailment
Config: depth-1

Question: Dave is not quiet.
Label: entailment
Config: depth-1

Question: Anne is rough.
Label: not entailment
Config: depth-1

Question: Fiona is not smart.
Label: entailment
Config: depth-1

Question: Dave is kind.
Label: not entailment
Config: depth-1

Question: Fiona is white.
Label: entailment
Config: depth-1

Question: Fiona is not white.
Label: not entailment
Config: depth-1


In [19]:

# Inspect the same example in detail

example = dataset["train"][0]

print("Context:")
for sentence in example["context"].split("."):
    if sentence.strip():
        print("-", sentence.strip())

print("\nQuestion:", example["question"])
print("Label:", example["label"])
print("Configuration:", example["config"])


Context:
- Anne is quiet
- Anne is not young
- Bob is kind
- Bob is young
- Dave is rough
- Dave is round
- Dave is smart
- Dave is not young
- Fiona is quiet
- Fiona is not round
- Kind, young things are not smart

Question: Bob is kind.
Label: entailment
Configuration: depth-1


In [23]:

import re
def parse_context(context):
    facts = set()
    rules = []
    unsupported = []

    for sentence in context.split("."):
        sentence = sentence.strip()

        if not sentence:
            continue

        rule = parse_rule(sentence)

        if rule is not None:
            rules.append(rule)
            continue

        fact = parse_atom(sentence)

        if fact is not None:
            facts.add(fact)
        else:
            unsupported.append(sentence)

    return facts, rules, unsupported


# Test on actual RuleTaker context
example = dataset["train"][0]

facts, rules, unsupported = parse_context(example["context"])

print("EXTRACTED FACTS:")
for fact in sorted(facts):
    print(fact)

print("\nEXTRACTED RULES:")
for rule in rules:
    print(rule)


EXTRACTED FACTS:
('kind', ('bob',))
('not_round', ('fiona',))
('not_young', ('anne',))
('not_young', ('dave',))
('quiet', ('anne',))
('quiet', ('fiona',))
('rough', ('dave',))
('round', ('dave',))
('smart', ('dave',))
('young', ('bob',))

EXTRACTED RULES:
([('kind', ('?x',)), ('young', ('?x',))], ('not_smart', ('?x',)))


In [133]:

def parse_atom(sentence):
    sentence = sentence.strip().rstrip(".")

    # Standardize variables
    sentence = re.sub(
        r"\b(someone|something|they|it)\b",
        "?x",
        sentence,
        flags=re.I
    )


    # Negative property
    m = re.fullmatch(
        r"(?:the\s+)?(\?x|\w+) is not (\w+)",
        sentence, re.I
    )
    if m:
        entity, prop = m.groups()
        return ("not_" + prop.lower(), (entity.lower(),))

    # Positive property: is or are
    m = re.fullmatch(
        r"(?:the\s+)?(\?x|\w+) (?:is|are) (\w+)",
        sentence, re.I
    )
    if m:
        entity, prop = m.groups()
        return (prop.lower(), (entity.lower(),))

    # Negative relationship
    m = re.fullmatch(
        r"(?:the\s+)?(\?x|\w+) does not (\w+) (?:the\s+)?(\w+)",
        sentence, re.I
    )
    if m:
        subject, verb, obj = m.groups()
        return (
            "not_" + normalize_verb(verb),
            (subject.lower(), obj.lower())
        )

    # Positive relationship
    m = re.fullmatch(
        r"(?:the\s+)?(\?x|\w+) (\w+) (?:the\s+)?(\w+)",
        sentence, re.I
    )
    if m:
        subject, verb, obj = m.groups()
        return (
            normalize_verb(verb),
            (subject.lower(), obj.lower())
        )

    return None



def parse_rule(sentence):
    sentence = sentence.strip().rstrip(".")

    # Standardize variables
    sentence = re.sub(
        r"\b(someone|something|they|it)\b",
        "?x", sentence, flags=re.I
    )

    # Handle universal rules
    m = re.fullmatch(
        r"All\s+(\w+)\s+things\s+(?:are|is)\s+(not\s+)?(\w+)",
        sentence, re.I
    )

    if m:
        condition, negation, conclusion = m.groups()
        predicate = conclusion.lower()

        if negation:
            predicate = "not_" + predicate

        return (
            [(condition.lower(), ("?x",))],
            (predicate, ("?x",))
        )

    # Your remaining existing rule patterns continue here...

    # Blue, rough people are green
    m = re.fullmatch(
        r"(\w+),\s*(\w+)\s+people\s+(?:are|is)\s+(\w+)",
        sentence, re.I
    )
    if m:
        p1, p2, conclusion = m.groups()
        return (
            [(p1.lower(), ("?x",)), (p2.lower(), ("?x",))],
            (conclusion.lower(), ("?x",))
        )

    # Kind, young things are not smart
    m = re.fullmatch(
        r"(\w+),\s*(\w+)\s+things\s+are\s+not\s+(\w+)",
        sentence, re.I
    )
    if m:
        p1, p2, conclusion = m.groups()
        return (
            [(p1.lower(), ("?x",)), (p2.lower(), ("?x",))],
            ("not_" + conclusion.lower(), ("?x",))
        )

    # If ... then ...
    m = re.fullmatch(r"If (.+?) then (.+)", sentence, re.I)
    if not m:
        return None

    condition_text, conclusion_text = m.groups()

    parts = re.split(r"\s+and\s+", condition_text, flags=re.I)
    conditions = []

    for i, part in enumerate(parts):
        part = part.strip()

        # Handle shortened conditions such as "and not rough"
        if i > 0 and re.fullmatch(r"(?:not\s+)?\w+", part, re.I):
            part = "?x is " + part

        atom = parse_atom(part)

        if atom is None:
            return None

        conditions.append(atom)

    conclusion = parse_atom(conclusion_text.strip())

    if conclusion is None:
        return None

    return conditions, conclusion




def parse_context(context):

    facts = set()
    rules = []
    unsupported = []

    sentences = [
        s.strip()
        for s in context.split(".")
        if s.strip()
    ]

    for sentence in sentences:

        # Try parsing as a rule first
        rule = parse_rule(sentence)

        if rule is not None:
            rules.append(rule)
            continue

        # Otherwise, try parsing as a fact
        fact = parse_atom(sentence)

        if fact is not None:
            facts.add(fact)
        else:
            unsupported.append(sentence)

    return facts, rules, unsupported


# Test on actual RuleTaker example
example = dataset["train"][0]

facts, rules, unsupported = parse_context(
    example["context"]
)

print("FACTS:")
for fact in sorted(facts):
    print(fact)

print("\nRULES:")
for rule in rules:
    print(rule)

print("\nUNSUPPORTED SENTENCES:")
for sentence in unsupported:
    print(sentence)


FACTS:
('kind', ('bob',))
('not_round', ('fiona',))
('not_young', ('anne',))
('not_young', ('dave',))
('quiet', ('anne',))
('quiet', ('fiona',))
('rough', ('dave',))
('round', ('dave',))
('smart', ('dave',))
('young', ('bob',))

RULES:
([('kind', ('?x',)), ('young', ('?x',))], ('not_smart', ('?x',)))

UNSUPPORTED SENTENCES:


In [134]:
print(parse_atom("?x do not like the bald eagle"))

None


In [108]:
print(parse_rule("All blue things are white."))
print(parse_rule("All quiet things are not young."))

([('blue', ('?x',))], ('white', ('?x',)))
([('quiet', ('?x',))], ('not_young', ('?x',)))


In [42]:

example = dataset["train"][0]

facts, rules, unsupported = parse_context(
    example["context"]
)

print("Number of facts:", len(facts))
print("Number of rules:", len(rules))
print("Unsupported sentences:", len(unsupported))

print("\nTest relationship query:")
print(parse_atom("The cow likes the tiger."))

print("\nTest property query:")
print(parse_atom("The cat is not cold."))

print("\nTest relationship rule:")
print(parse_rule(
    "If someone is young then they chase the lion."
))


Number of facts: 10
Number of rules: 1
Unsupported sentences: 0

Test relationship query:
('likes', ('cow', 'tiger'))

Test property query:
('not_cold', ('cat',))

Test relationship rule:
([('young', ('?x',))], ('chases', ('?x', 'lion')))


In [39]:

# Step 1: Forward chaining

def match_condition(condition, facts, substitution):
    predicate, arguments = condition
    matches = []

    for fact_predicate, fact_arguments in facts:
        if predicate != fact_predicate:
            continue

        if len(arguments) != len(fact_arguments):
            continue

        updated = substitution.copy()
        valid = True

        for pattern, value in zip(arguments, fact_arguments):
            if pattern.startswith("?"):
                if pattern in updated and updated[pattern] != value:
                    valid = False
                    break
                updated[pattern] = value
            elif pattern != value:
                valid = False
                break

        if valid:
            matches.append(updated)

    return matches


def forward_chaining(facts, rules):
    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:
            substitutions = [{}]

            for condition in conditions:
                new_substitutions = []

                for sub in substitutions:
                    new_substitutions.extend(
                        match_condition(condition, known_facts, sub)
                    )

                substitutions = new_substitutions

                if not substitutions:
                    break

            for sub in substitutions:
                predicate, arguments = conclusion

                derived = (
                    predicate,
                    tuple(
                        sub[arg] if arg.startswith("?") else arg
                        for arg in arguments
                    )
                )

                if derived not in known_facts:
                    known_facts.add(derived)
                    changed = True

    return known_facts


# Step 2: Parse the question

def parse_question(question):
    question = question.strip().rstrip(".")

    match = re.fullmatch(
        r"([A-Za-z]+) is not ([A-Za-z]+)",
        question,
        re.IGNORECASE
    )

    if match:
        entity, predicate = match.groups()
        return ("not_" + predicate.lower(), (entity,))

    match = re.fullmatch(
        r"([A-Za-z]+) is ([A-Za-z]+)",
        question,
        re.IGNORECASE
    )

    if match:
        entity, predicate = match.groups()
        return (predicate.lower(), (entity,))

    return None


# Step 3: Run on an actual RuleTaker example

example = dataset["train"][0]

facts, rules, unsupported = parse_context(example["context"])

derived_facts = forward_chaining(facts, rules)

query = parse_question(example["question"])

print("QUESTION:", example["question"])
print("EXPECTED LABEL:", example["label"])

print("\nDERIVED FACTS:")
for fact in sorted(derived_facts):
    print(fact)

print("\nPARSED QUERY:", query)
print("QUERY FOUND:", query in derived_facts)


QUESTION: Bob is kind.
EXPECTED LABEL: entailment

DERIVED FACTS:
('kind', ('bob',))
('not_round', ('fiona',))
('not_smart', ('bob',))
('not_young', ('anne',))
('not_young', ('dave',))
('quiet', ('anne',))
('quiet', ('fiona',))
('rough', ('dave',))
('round', ('dave',))
('smart', ('dave',))
('young', ('bob',))

PARSED QUERY: ('kind', ('Bob',))
QUERY FOUND: False


In [25]:
def normalize_verb(verb):
    verb = verb.lower()

    if verb in ["chase", "chases"]:
        return "chases"
    if verb in ["see", "sees"]:
        return "sees"
    if verb in ["need", "needs"]:
        return "needs"
    if verb in ["like", "likes"]:
        return "likes"

    return verb

In [26]:
print(normalize_verb("likes"))
print(normalize_verb("sees"))
print(parse_atom("The cow likes the tiger."))

likes
sees
('likes', ('cow', 'tiger'))


In [136]:

import re



def parse_atom(sentence):
    sentence = sentence.strip().rstrip(".")

    sentence = re.sub(
        r"\b(someone|something|they|it)\b",
        "?x", sentence, flags=re.I
    )

    sentence = re.sub(r"^the\s+", "", sentence, flags=re.I)

    verbs = [
        "chase", "chases", "see", "sees",
        "like", "likes", "need", "needs",
        "have", "has", "eat", "eats",
        "visit", "visits", "help", "helps"
    ]

    verb_pattern = "|".join(sorted(verbs, key=len, reverse=True))

    # Negative property
    m = re.fullmatch(
        r"(.+?)\s+(?:is|are)\s+not\s+(\w+)", sentence, re.I
    )
    if m:
        entity, prop = m.groups()
        return ("not_" + prop.lower(), (entity.lower(),))

    # Positive property
    m = re.fullmatch(
        r"(.+?)\s+(?:is|are)\s+(\w+)", sentence, re.I
    )
    if m:
        entity, prop = m.groups()
        return (prop.lower(), (entity.lower(),))

    # Negative relationship
    m = re.fullmatch(
        rf"(.+?)\s+do(?:es)? not\s+({verb_pattern})\s+(.+)",
        sentence, re.I
    )
    if m:
        subject, verb, obj = m.groups()
        obj = re.sub(r"^the\s+", "", obj, flags=re.I)
        return (
            "not_" + normalize_verb(verb),
            (subject.lower().strip(), obj.lower().strip())
        )

    # Positive relationship
    m = re.fullmatch(
        rf"(.+?)\s+({verb_pattern})\s+(.+)",
        sentence, re.I
    )
    if m:
        subject, verb, obj = m.groups()
        obj = re.sub(r"^the\s+", "", obj, flags=re.I)
        return (
            normalize_verb(verb),
            (subject.lower().strip(), obj.lower().strip())
        )

    return None


def parse_rule(sentence):
    sentence = sentence.strip().rstrip(".")

    sentence = re.sub(
        r"\b(someone|something|they|it)\b",
        "?x", sentence, flags=re.I
    )

    # All blue things are white
    m = re.fullmatch(
        r"All\s+(\w+)\s+things\s+(?:are|is)\s+(not\s+)?(\w+)",
        sentence, re.I
    )
    if m:
        condition, negation, conclusion = m.groups()
        predicate = conclusion.lower()
        if negation:
            predicate = "not_" + predicate
        return (
            [(condition.lower(), ("?x",))],
            (predicate, ("?x",))
        )

    # All red people are smart
    m = re.fullmatch(
        r"All\s+(\w+)\s+people\s+(?:are|is)\s+(not\s+)?(\w+)",
        sentence, re.I
    )
    if m:
        condition, negation, conclusion = m.groups()
        predicate = conclusion.lower()
        if negation:
            predicate = "not_" + predicate
        return (
            [(condition.lower(), ("?x",))],
            (predicate, ("?x",))
        )

    # Blue, rough people are green
    m = re.fullmatch(
        r"(\w+),\s*(\w+)\s+people\s+(?:are|is)\s+(not\s+)?(\w+)",
        sentence, re.I
    )
    if m:
        p1, p2, negation, conclusion = m.groups()
        predicate = conclusion.lower()
        if negation:
            predicate = "not_" + predicate
        return (
            [(p1.lower(), ("?x",)), (p2.lower(), ("?x",))],
            (predicate, ("?x",))
        )

    # If ... then ...
    m = re.fullmatch(r"If (.+?) then (.+)", sentence, re.I)

    if not m:
        return None

    condition_text, conclusion_text = m.groups()

    parts = re.split(r"\s+and\s+", condition_text, flags=re.I)
    conditions = []

    for i, part in enumerate(parts):
        part = part.strip()

        if i > 0 and re.fullmatch(r"(?:not\s+)?\w+", part, re.I):
            part = "?x is " + part

        atom = parse_atom(part)

        if atom is None:
            return None

        conditions.append(atom)

    conclusion = parse_atom(conclusion_text.strip())

    if conclusion is None:
        return None

    return conditions, conclusion



def parse_context(context):
    facts = set()
    rules = []
    unsupported = []

    for sentence in context.split("."):
        sentence = sentence.strip()

        if not sentence:
            continue

        rule = parse_rule(sentence)

        if rule is not None:
            rules.append(rule)
            continue

        fact = parse_atom(sentence)

        if fact is not None:
            facts.add(fact)
        else:
            unsupported.append(sentence)

    return facts, rules, unsupported


def parse_question(question):
    return parse_atom(question)


In [137]:
print(parse_atom("?x do not like the bald eagle"))

('not_likes', ('?x', 'bald eagle'))


In [43]:
print(parse_atom("The cow does not need the cat."))

('not_needs', ('cow', 'cat'))


In [44]:

# Test 1: Negative property
print(parse_atom("The cat is not cold."))

# Test 2: Positive relationship
print(parse_atom("The cow likes the tiger."))

# Test 3: Negative relationship
print(parse_atom("The cow does not chase the cat."))

# Test 4: Single-condition rule
print(parse_rule(
    "If someone is young then they chase the lion."
))

# Test 5: Multiple-condition rule
print(parse_rule(
    "If someone likes the cow and the cow chases the lion "
    "then the cow chases the tiger."
))

# Test 6: Actual dataset context
example = dataset["train"][0]

facts, rules, unsupported = parse_context(
    example["context"]
)

print("\nDataset inspection:")
print("Facts:", len(facts))
print("Rules:", len(rules))
print("Unsupported:", unsupported)


('not_cold', ('cat',))
('likes', ('cow', 'tiger'))
('not_chases', ('cow', 'cat'))
([('young', ('?x',))], ('chases', ('?x', 'lion')))
([('likes', ('?x', 'cow')), ('chases', ('cow', 'lion'))], ('chases', ('cow', 'tiger')))

Dataset inspection:
Facts: 10
Rules: 1
Unsupported: []


In [45]:

print(parse_rule(
    "If someone is young then they chase the lion."
))

print(parse_rule(
    "If someone likes the cow and the cow chases the lion "
    "then the cow chases the tiger."
))

print(parse_atom("Bob is young."))
print(parse_atom("The cat is not cold."))


([('young', ('?x',))], ('chases', ('?x', 'lion')))
([('likes', ('?x', 'cow')), ('chases', ('cow', 'lion'))], ('chases', ('cow', 'tiger')))
('young', ('bob',))
('not_cold', ('cat',))


In [46]:
context = """
Anne is quiet.
Anne is not young.
Bob is kind.
Bob is young.
Dave is rough.
Dave is round.
Dave is smart.
Dave is not young.
Fiona is quiet.
Fiona is not round.
Kind, young things are not smart.
"""

facts, rules, unsupported = parse_context(context)

print("FACTS:")
for fact in sorted(facts):
    print(fact)

print("\nRULES:")
for rule in rules:
    print(rule)

print("\nUNSUPPORTED:")
print(unsupported)

FACTS:
('kind', ('bob',))
('not_round', ('fiona',))
('not_young', ('anne',))
('not_young', ('dave',))
('quiet', ('anne',))
('quiet', ('fiona',))
('rough', ('dave',))
('round', ('dave',))
('smart', ('dave',))
('young', ('bob',))

RULES:
([('kind', ('?x',)), ('young', ('?x',))], ('not_smart', ('?x',)))

UNSUPPORTED:
[]


In [47]:
new_facts = forward_chaining(facts, rules)

print("New facts derived:")
for fact in sorted(new_facts - facts):
    print(fact)

print("\nIs Bob not smart?")
print(("not_smart", ("bob",)) in new_facts)

New facts derived:
('not_smart', ('bob',))

Is Bob not smart?
True


In [48]:
test_sentences = [
    "Bob is kind.",
    "Bob is not smart.",
    "Anne is quiet.",
    "The cow likes the tiger.",
    "The cow does not chase the cat."
]

for sentence in test_sentences:
    print(sentence, "->", parse_atom(sentence))

Bob is kind. -> ('kind', ('bob',))
Bob is not smart. -> ('not_smart', ('bob',))
Anne is quiet. -> ('quiet', ('anne',))
The cow likes the tiger. -> ('likes', ('cow', 'tiger'))
The cow does not chase the cat. -> ('not_chases', ('cow', 'cat'))


In [49]:
# Select 5 examples from the development dataset
for i in range(5):
    example = dataset["dev"][i]

    context = example["context"]
    question = example["question"]
    label = example["label"]

    facts, rules, unsupported = parse_context(context)
    all_facts = forward_chaining(facts, rules)

    query = parse_atom(question)

    print("\nExample:", i + 1)
    print("Question:", question)
    print("Dataset label:", label)
    print("Parsed query:", query)
    print("Query found in facts:", query in all_facts if query else "Could not parse")
    print("Unsupported sentences:", unsupported)


Example: 1
Question: The cow likes the tiger.
Dataset label: entailment
Parsed query: ('likes', ('cow', 'tiger'))
Query found in facts: True
Unsupported sentences: []

Example: 2
Question: The cow does not need the cat.
Dataset label: not entailment
Parsed query: ('not_needs', ('cow', 'cat'))
Query found in facts: False
Unsupported sentences: []

Example: 3
Question: The lion needs the tiger.
Dataset label: entailment
Parsed query: ('needs', ('lion', 'tiger'))
Query found in facts: True
Unsupported sentences: []

Example: 4
Question: The cat does not chase the lion.
Dataset label: not entailment
Parsed query: ('not_chases', ('cat', 'lion'))
Query found in facts: False
Unsupported sentences: []

Example: 5
Question: The cat is not cold.
Dataset label: entailment
Parsed query: ('not_cold', ('cat',))
Query found in facts: False
Unsupported sentences: []


In [50]:
example = dataset["dev"][4]

print("CONTEXT:")
print(example["context"])

print("\nQUESTION:")
print(example["question"])

print("\nLABEL:")
print(example["label"])

facts, rules, unsupported = parse_context(example["context"])

print("\nPARSED FACTS:")
for fact in sorted(facts):
    print(fact)

print("\nPARSED RULES:")
for rule in rules:
    print(rule)

print("\nUNSUPPORTED:")
print(unsupported)

CONTEXT:
The cat chases the cow. The cat is big. The cat is red. The cat is round. The cat is young. The cat needs the tiger. The cow chases the lion. The cow likes the tiger. The cow needs the cat. The lion chases the tiger. The lion is young. The tiger needs the lion. If someone likes the cat then they are young. If the cow chases the lion then the lion needs the tiger. If someone chases the cat then they are cold. If someone is young then they chase the lion. If someone likes the cow and the cow chases the lion then the cow chases the tiger. If someone likes the lion and they chase the lion then the lion is cold. If someone likes the tiger and the tiger needs the cow then they chase the cow. If someone needs the cat and the cat likes the tiger then they are young.

QUESTION:
The cat is not cold.

LABEL:
entailment

PARSED FACTS:
('big', ('cat',))
('chases', ('cat', 'cow'))
('chases', ('cow', 'lion'))
('chases', ('lion', 'tiger'))
('likes', ('cow', 'tiger'))
('needs', ('cat', 'tiger'

In [51]:
print(parse_atom("They are young"))
print(parse_atom("They are cold"))
print(parse_atom("They are not cold"))
print(parse_atom("The cat is not cold"))

('young', ('?x',))
('cold', ('?x',))
('not_cold', ('?x',))
('not_cold', ('cat',))


In [52]:
example = dataset["dev"][4]

facts, rules, unsupported = parse_context(example["context"])

all_facts = forward_chaining(facts, rules)

print("Question:", example["question"])
print("Dataset label:", example["label"])

print("\nDerived facts:")
for fact in sorted(all_facts - facts):
    print(fact)

print("\nIs cat cold?", ("cold", ("cat",)) in all_facts)
print("Is cat not cold?", ("not_cold", ("cat",)) in all_facts)

Question: The cat is not cold.
Dataset label: entailment

Derived facts:
('chases', ('cat', 'lion'))
('chases', ('lion', 'lion'))
('needs', ('lion', 'tiger'))

Is cat cold? False
Is cat not cold? False


In [53]:
print(("chases", ("cat", "cat")) in all_facts)
print(("chases", ("cat", "cow")) in all_facts)

False
True


In [54]:
example = dataset["dev"][4]

print("Question:", example["question"])
print("Label:", example["label"])

print("\nAll context sentences:")
for sentence in example["context"].split("."):
    if sentence.strip():
        print("-", sentence.strip())

Question: The cat is not cold.
Label: entailment

All context sentences:
- The cat chases the cow
- The cat is big
- The cat is red
- The cat is round
- The cat is young
- The cat needs the tiger
- The cow chases the lion
- The cow likes the tiger
- The cow needs the cat
- The lion chases the tiger
- The lion is young
- The tiger needs the lion
- If someone likes the cat then they are young
- If the cow chases the lion then the lion needs the tiger
- If someone chases the cat then they are cold
- If someone is young then they chase the lion
- If someone likes the cow and the cow chases the lion then the cow chases the tiger
- If someone likes the lion and they chase the lion then the lion is cold
- If someone likes the tiger and the tiger needs the cow then they chase the cow
- If someone needs the cat and the cat likes the tiger then they are young


In [55]:
print(dataset)
print("\nFeatures:")
print(dataset["dev"].features)

print("\nExample keys:")
print(dataset["dev"][4].keys())

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 480152
    })
    test: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 151911
    })
    dev: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 75872
    })
})

Features:
{'context': Value('string'), 'question': Value('string'), 'label': Value('string'), 'config': Value('string')}

Example keys:
dict_keys(['context', 'question', 'label', 'config'])


In [56]:
example = dataset["dev"][4]

print("Configuration:", example["config"])
print("Question:", example["question"])
print("Label:", example["label"])

Configuration: depth-1
Question: The cat is not cold.
Label: entailment


In [57]:
print("Cat is young:",
      ("young", ("cat",)) in all_facts)

print("Cat chases lion:",
      ("chases", ("cat", "lion")) in all_facts)

print("Cat chases cat:",
      ("chases", ("cat", "cat")) in all_facts)

print("Cat is cold:",
      ("cold", ("cat",)) in all_facts)

Cat is young: True
Cat chases lion: True
Cat chases cat: False
Cat is cold: False


In [58]:
print("Dataset:", dataset)
print("Example config:", dataset["dev"][4]["config"])

print("\nOther examples with the same context:")
target_context = dataset["dev"][4]["context"]

count = 0
for item in dataset["dev"]:
    if item["context"] == target_context:
        print(item["question"], "=>", item["label"])
        count += 1
        if count == 10:
            break

Dataset: DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 480152
    })
    test: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 151911
    })
    dev: Dataset({
        features: ['context', 'question', 'label', 'config'],
        num_rows: 75872
    })
})
Example config: depth-1

Other examples with the same context:
The cow likes the tiger. => entailment
The cow does not need the cat. => not entailment
The lion needs the tiger. => entailment
The cat does not chase the lion. => not entailment
The cat is not cold. => entailment
The cow is cold. => not entailment
The cow does not chase the cat. => entailment
The cat chases the tiger. => not entailment


In [59]:
print("Cow needs cat:",
      ("needs", ("cow", "cat")) in all_facts)

print("Cow does not need cat:",
      ("not_needs", ("cow", "cat")) in all_facts)

print("Cow chases cat:",
      ("chases", ("cow", "cat")) in all_facts)

print("Cow does not chase cat:",
      ("not_chases", ("cow", "cat")) in all_facts)

Cow needs cat: True
Cow does not need cat: False
Cow chases cat: False
Cow does not chase cat: False


In [60]:
def answer_query(query, all_facts):

    if query is None:
        return "unknown"

    predicate, arguments = query

    # If the exact query exists
    if query in all_facts:
        return "entailment"

    # Handle negative queries
    if predicate.startswith("not_"):
        positive_predicate = predicate[4:]
        positive_query = (positive_predicate, arguments)

        if positive_query in all_facts:
            return "not entailment"

        return "entailment"

    # Handle positive queries
    negative_predicate = "not_" + predicate
    negative_query = (negative_predicate, arguments)

    if negative_query in all_facts:
        return "not entailment"

    return "not entailment"

In [61]:
print(answer_query(
    ("not_chases", ("cow", "cat")),
    all_facts
))

print(answer_query(
    ("not_needs", ("cow", "cat")),
    all_facts
))

entailment
not entailment


In [62]:
target_context = dataset["dev"][4]["context"]

for item in dataset["dev"]:
    if item["context"] == target_context:

        facts, rules, unsupported = parse_context(item["context"])
        all_facts = forward_chaining(facts, rules)

        query = parse_atom(item["question"])
        prediction = answer_query(query, all_facts)

        print("Question:", item["question"])
        print("Actual:", item["label"])
        print("Predicted:", prediction)
        print("-" * 50)

Question: The cow likes the tiger.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cow does not need the cat.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The lion needs the tiger.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cat does not chase the lion.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The cat is not cold.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cow is cold.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The cow does not chase the cat.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cat chases the tiger.
Actual: not entailment
Predicted: not ent

In [63]:
target_context = dataset["dev"][4]["context"]

facts, rules, unsupported = parse_context(target_context)
all_facts = forward_chaining(facts, rules)

for item in dataset["dev"]:
    if item["context"] == target_context:
        query = parse_atom(item["question"])
        prediction = answer_query(query, all_facts)

        print("Question:", item["question"])
        print("Actual:", item["label"])
        print("Predicted:", prediction)
        print("-" * 50)

Question: The cow likes the tiger.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cow does not need the cat.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The lion needs the tiger.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cat does not chase the lion.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The cat is not cold.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cow is cold.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The cow does not chase the cat.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cat chases the tiger.
Actual: not entailment
Predicted: not ent

In [64]:
print("Positive fact exists:",
      ("needs", ("cow", "cat")) in all_facts)

print("Negative query:",
      ("not_needs", ("cow", "cat")))

print("Direct function result:",
      answer_query(("not_needs", ("cow", "cat")), all_facts))

Positive fact exists: True
Negative query: ('not_needs', ('cow', 'cat'))
Direct function result: not entailment


In [65]:
target_context = dataset["dev"][4]["context"]

facts, rules, unsupported = parse_context(target_context)
all_facts = forward_chaining(facts, rules)

for item in dataset["dev"]:
    if item["context"] == target_context:

        query = parse_atom(item["question"])
        prediction = answer_query(query, all_facts)

        print("Question:", item["question"])
        print("Actual:", item["label"])
        print("Predicted:", prediction)
        print("-" * 50)

Question: The cow likes the tiger.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cow does not need the cat.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The lion needs the tiger.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cat does not chase the lion.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The cat is not cold.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cow is cold.
Actual: not entailment
Predicted: not entailment
--------------------------------------------------
Question: The cow does not chase the cat.
Actual: entailment
Predicted: entailment
--------------------------------------------------
Question: The cat chases the tiger.
Actual: not entailment
Predicted: not ent

In [66]:
target_context = dataset["dev"][4]["context"]

facts, rules, unsupported = parse_context(target_context)
all_facts = forward_chaining(facts, rules)

for item in dataset["dev"]:
    if item["context"] == target_context:
        if "does not need the cat" in item["question"]:

            query = parse_atom(item["question"])

            print("Parsed query:", query)
            print("Positive fact exists:",
                  ("needs", ("cow", "cat")) in all_facts)
            print("Negative query exists:",
                  query in all_facts)
            print("Prediction:",
                  answer_query(query, all_facts))

Parsed query: ('not_needs', ('cow', 'cat'))
Positive fact exists: True
Negative query exists: False
Prediction: not entailment


In [67]:
print(normalize_verb("need"))
print(normalize_verb("needs"))
print(parse_atom("The cow does not need the cat."))

needs
needs
('not_needs', ('cow', 'cat'))


In [121]:
correct = 0
total = 100
errors = []

for i in range(total):
    example = dataset["dev"][i]

    facts, rules, unsupported = parse_context(example["context"])
    all_facts = forward_chaining(facts, rules)

    query = parse_atom(example["question"])
    prediction = answer_query(query, all_facts)

    actual = example["label"]

    if prediction == actual:
        correct += 1
    else:
        errors.append({
            "question": example["question"],
            "actual": actual,
            "predicted": prediction,
            "context": example["context"]
        })

accuracy = (correct / total) * 100

print("Total examples:", total)
print("Correct predictions:", correct)
print("Incorrect predictions:", total - correct)
print("Accuracy:", accuracy, "%")

print("\nFirst 5 errors:")
for error in errors[:5]:
    print(error["question"])
    print("Actual:", error["actual"])
    print("Predicted:", error["predicted"])
    print("-" * 40)

Total examples: 100
Correct predictions: 100
Incorrect predictions: 0
Accuracy: 100.0 %

First 5 errors:


In [69]:
for i, error in enumerate(errors[:5]):

    print("\n========== ERROR", i + 1, "==========")

    print("\nQUESTION:", error["question"])
    print("ACTUAL:", error["actual"])
    print("PREDICTED:", error["predicted"])

    facts, rules, unsupported = parse_context(error["context"])
    all_facts = forward_chaining(facts, rules)

    print("\nPARSED FACTS:")
    for fact in sorted(facts):
        print(fact)

    print("\nPARSED RULES:")
    for rule in rules:
        print(rule)

    print("\nUNSUPPORTED SENTENCES:")
    print(unsupported)

    print("\nDERIVED FACTS:")
    for fact in sorted(all_facts - facts):
        print(fact)


========== ERROR 1 ==========

QUESTION: Bob is nice.
ACTUAL: entailment
PREDICTED: not entailment

PARSED FACTS:
('not_red', ('bob',))
('not_round', ('harry',))
('rough', ('gary',))
('young', ('erin',))

PARSED RULES:
([('not_round', ('bob',)), ('not_young', ('bob',))], ('nice', ('bob',)))
([('red', ('?x',))], ('not_smart', ('?x',)))

UNSUPPORTED SENTENCES:
[]

DERIVED FACTS:

========== ERROR 2 ==========

QUESTION: Bob is not nice.
ACTUAL: not entailment
PREDICTED: entailment

PARSED FACTS:
('not_red', ('bob',))
('not_round', ('harry',))
('rough', ('gary',))
('young', ('erin',))

PARSED RULES:
([('not_round', ('bob',)), ('not_young', ('bob',))], ('nice', ('bob',)))
([('red', ('?x',))], ('not_smart', ('?x',)))

UNSUPPORTED SENTENCES:
[]

DERIVED FACTS:

========== ERROR 3 ==========

QUESTION: Charlie is big.
ACTUAL: entailment
PREDICTED: not entailment

PARSED FACTS:
('big', ('dave',))
('big', ('gary',))
('blue', ('charlie',))
('smart', ('harry',))

PARSED RULES:
([('round', ('?x',

In [70]:
print(parse_atom("They are furry"))
print(parse_atom("They are cold"))
print(parse_atom("If someone is young then they chase the lion"))
print(parse_atom("The cat is not cold"))

('furry', ('?x',))
('cold', ('?x',))
('chases', ('if ?x is young then ?x', 'lion'))
('not_cold', ('cat',))


In [32]:
import inspect

print(inspect.getsource(parse_rule))

def parse_rule(sentence):
    sentence = sentence.strip().rstrip(".")

    # Standardize variables
    sentence = re.sub(
        r"\b(someone|something|they|it)\b",
        "?x", sentence, flags=re.I
    )

    # Pattern 1: All red people are not smart
    m = re.fullmatch(
        r"All\s+(\w+)\s+people\s+(?:are|is)\s+not\s+(\w+)",
        sentence, re.I
    )
    if m:
        p1, conclusion = m.groups()
        return (
            [(p1.lower(), ("?x",))],
            ("not_" + conclusion.lower(), ("?x",))
        )

    # Pattern 2: White things are green / Cold things are quiet
    m = re.fullmatch(
        r"(\w+)\s+things\s+(?:are|is)\s+(\w+)",
        sentence, re.I
    )
    if m:
        condition, conclusion = m.groups()
        return (
            [(condition.lower(), ("?x",))],
            (conclusion.lower(), ("?x",))
        )

    # Pattern 3: Blue, rough people are green
    m = re.fullmatch(
        r"(\w+),\s*(\w+)\s+people\s+(?:are|is)\s+(\w+)",
        sentence, 

In [71]:
print(parse_rule(
    "If someone is white and rough then they are big."
))

print(parse_rule(
    "Blue, rough people are green."
))

print(parse_rule(
    "If someone is young then they chase the lion."
))

([('white', ('?x',)), ('rough', ('?x',))], ('big', ('?x',)))
([('blue', ('?x',)), ('rough', ('?x',))], ('green', ('?x',)))
([('young', ('?x',))], ('chases', ('?x', 'lion')))


In [72]:
print(parse_rule(
    "If someone is white and rough then they are big."
))

([('white', ('?x',)), ('rough', ('?x',))], ('big', ('?x',)))


In [34]:
import re

sentence = "If someone is white and rough then they are big."

sentence = re.sub(
    r"\b(someone|something|they|it)\b",
    "?x",
    sentence,
    flags=re.I
)

print(sentence)

print(parse_atom("?x is white"))
print(parse_atom("?x is rough"))
print(parse_atom("?x are big"))

If ?x is white and rough then ?x are big.
('white', ('?x',))
('rough', ('?x',))
('big', ('?x',))


In [73]:
print(parse_atom("?x are big"))
print(parse_atom("?x are furry"))
print(parse_rule(
    "If someone is white and rough then they are big."
))

('big', ('?x',))
('furry', ('?x',))
([('white', ('?x',)), ('rough', ('?x',))], ('big', ('?x',)))


In [74]:
sentence = "If someone is white and rough then they are big."

sentence = re.sub(
    r"\b(someone|something|they|it)\b",
    "?x",
    sentence,
    flags=re.I
)

print("Standardized:", sentence)

m = re.fullmatch(r"If (.+?) then (.+)", sentence, re.I)

print("Conditions:", m.group(1))
print("Conclusion:", m.group(2))

print("First condition:", parse_atom("?x is white"))
print("Second condition:", parse_atom("?x is rough"))
print("Conclusion:", parse_atom("?x are big"))

Standardized: If ?x is white and rough then ?x are big.
Conditions: ?x is white and rough
Conclusion: ?x are big.
First condition: ('white', ('?x',))
Second condition: ('rough', ('?x',))
Conclusion: ('big', ('?x',))


In [75]:
print(parse_rule(
    "If someone is white and rough then they are big."
))

([('white', ('?x',)), ('rough', ('?x',))], ('big', ('?x',)))


In [35]:
import inspect

print(inspect.getsource(parse_rule))

sentence = "If someone is white and rough then they are big."

print("\nDirect parsing:")
print(parse_rule(sentence))

print("\nRegex test:")
print(bool(re.fullmatch(
    r"If (.+?) then (.+)",
    sentence,
    re.I
)))

def parse_rule(sentence):
    sentence = sentence.strip().rstrip(".")

    # Standardize variables
    sentence = re.sub(
        r"\b(someone|something|they|it)\b",
        "?x", sentence, flags=re.I
    )

    # Pattern 1: All red people are not smart
    m = re.fullmatch(
        r"All\s+(\w+)\s+people\s+(?:are|is)\s+not\s+(\w+)",
        sentence, re.I
    )
    if m:
        p1, conclusion = m.groups()
        return (
            [(p1.lower(), ("?x",))],
            ("not_" + conclusion.lower(), ("?x",))
        )

    # Pattern 2: White things are green / Cold things are quiet
    m = re.fullmatch(
        r"(\w+)\s+things\s+(?:are|is)\s+(\w+)",
        sentence, re.I
    )
    if m:
        condition, conclusion = m.groups()
        return (
            [(condition.lower(), ("?x",))],
            (conclusion.lower(), ("?x",))
        )

    # Pattern 3: Blue, rough people are green
    m = re.fullmatch(
        r"(\w+),\s*(\w+)\s+people\s+(?:are|is)\s+(\w+)",
        sentence, 

In [76]:
sentence = "If someone is white and rough then they are big."

sentence = re.sub(
    r"\b(someone|something|they|it)\b",
    "?x",
    sentence,
    flags=re.I
)

m = re.fullmatch(r"If (.+?) then (.+)", sentence, re.I)

condition_text, conclusion_text = m.groups()

print("Condition text:", condition_text)
print("Conclusion text:", conclusion_text)

parts = re.split(r"\s+and\s+", condition_text, flags=re.I)

print("\nParsed conditions:")
for part in parts:
    print(parse_atom(part.strip()))

print("\nParsed conclusion:")
print(parse_atom(conclusion_text.strip()))

Condition text: ?x is white and rough
Conclusion text: ?x are big.

Parsed conditions:
('white', ('?x',))
None

Parsed conclusion:
('big', ('?x',))


In [77]:
print(parse_rule(
    "If someone is white and rough then they are big."
))

([('white', ('?x',)), ('rough', ('?x',))], ('big', ('?x',)))


In [78]:
example = dataset["dev"][0]

facts, rules, unsupported = parse_context(example["context"])
all_facts = forward_chaining(facts, rules)

print("Question:", example["question"])
print("Actual:", example["label"])

print("\nParsed rules:")
for rule in rules:
    print(rule)

print("\nDerived facts:")
for fact in sorted(all_facts - facts):
    print(fact)

print("\nPrediction:",
      answer_query(parse_atom(example["question"]), all_facts))

Question: The cow likes the tiger.
Actual: entailment

Parsed rules:
([('likes', ('?x', 'cat'))], ('young', ('?x',)))
([('chases', ('cow', 'lion'))], ('needs', ('lion', 'tiger')))
([('chases', ('?x', 'cat'))], ('cold', ('?x',)))
([('young', ('?x',))], ('chases', ('?x', 'lion')))
([('likes', ('?x', 'cow')), ('chases', ('cow', 'lion'))], ('chases', ('cow', 'tiger')))
([('likes', ('?x', 'lion')), ('chases', ('?x', 'lion'))], ('cold', ('lion',)))
([('likes', ('?x', 'tiger')), ('needs', ('tiger', 'cow'))], ('chases', ('?x', 'cow')))
([('needs', ('?x', 'cat')), ('likes', ('cat', 'tiger'))], ('young', ('?x',)))

Derived facts:
('chases', ('cat', 'lion'))
('chases', ('lion', 'lion'))
('needs', ('lion', 'tiger'))

Prediction: entailment


In [79]:
# Select the first dev example
example = dataset["dev"][0]

facts, rules, unsupported = parse_context(example["context"])
all_facts = forward_chaining(facts, rules)

query = parse_atom(example["question"])
prediction = answer_query(query, all_facts)

print("Question:", example["question"])
print("Actual:", example["label"])
print("Predicted:", prediction)

print("\nUnsupported:", unsupported)

Question: The cow likes the tiger.
Actual: entailment
Predicted: entailment

Unsupported: []


In [163]:
correct = 0
total = 100
errors = []

for i in range(total):
    example = dataset["dev"][i]

    facts, rules, unsupported = parse_context(example["context"])
    all_facts = forward_chaining(facts, rules)

    query = parse_atom(example["question"])
    prediction = answer_query(query, all_facts)

    actual = example["label"]

    if prediction == actual:
        correct += 1
    else:
        errors.append({
            "question": example["question"],
            "actual": actual,
            "predicted": prediction,
            "context": example["context"]
        })

accuracy = correct / total * 100

print("Total examples:", total)
print("Correct:", correct)
print("Incorrect:", total - correct)
print("Accuracy:", accuracy, "%")

Total examples: 100
Correct: 94
Incorrect: 6
Accuracy: 94.0 %


In [81]:
print("Remaining errors:", len(errors))

for i, error in enumerate(errors[:10]):

    print("\nError:", i + 1)
    print("Question:", error["question"])
    print("Actual:", error["actual"])
    print("Predicted:", error["predicted"])

    facts, rules, unsupported = parse_context(error["context"])

    print("Unsupported sentences:", unsupported)
    print("Number of rules:", len(rules))

Remaining errors: 4

Error: 1
Question: Bob is nice.
Actual: entailment
Predicted: not entailment
Unsupported sentences: []
Number of rules: 2

Error: 2
Question: Bob is not nice.
Actual: not entailment
Predicted: entailment
Unsupported sentences: []
Number of rules: 2

Error: 3
Question: Charlie is big.
Actual: entailment
Predicted: not entailment
Unsupported sentences: []
Number of rules: 8

Error: 4
Question: Charlie is not big.
Actual: not entailment
Predicted: entailment
Unsupported sentences: []
Number of rules: 8


In [82]:
test_rules = [
    "All red people are not smart.",
    "White things are green.",
    "Cold things are quiet."
]

for rule in test_rules:
    print(rule)
    print(parse_rule(rule))
    print()

All red people are not smart.
([('red', ('?x',))], ('not_smart', ('?x',)))

White things are green.
([('white', ('?x',))], ('green', ('?x',)))

Cold things are quiet.
([('cold', ('?x',))], ('quiet', ('?x',)))



In [83]:
for rule in [
    "All red people are not smart.",
    "White things are green.",
    "Cold things are quiet."
]:
    print(parse_rule(rule))

([('red', ('?x',))], ('not_smart', ('?x',)))
([('white', ('?x',))], ('green', ('?x',)))
([('cold', ('?x',))], ('quiet', ('?x',)))


In [84]:
for i in range(100):
    example = dataset["dev"][i]

    if example["question"] == "Bob is nice.":
        facts, rules, unsupported = parse_context(example["context"])
        all_facts = forward_chaining(facts, rules)

        print("Question:", example["question"])
        print("Actual:", example["label"])

        print("\nParsed rules:")
        for rule in rules:
            print(rule)

        print("\nDerived facts:")
        for fact in sorted(all_facts - facts):
            print(fact)

        print("\nIs Bob nice?",
              ("nice", ("bob",)) in all_facts)

        break

Question: Bob is nice.
Actual: entailment

Parsed rules:
([('not_round', ('bob',)), ('not_young', ('bob',))], ('nice', ('bob',)))
([('red', ('?x',))], ('not_smart', ('?x',)))

Derived facts:

Is Bob nice? False


In [85]:

print("Bob not round:",
      ("not_round", ("bob",)) in facts)

print("Bob not young:",
      ("not_young", ("bob",)) in facts)

print("\nAll original facts:")
for fact in sorted(facts):
    print(fact)

Bob not round: False
Bob not young: False

All original facts:
('not_red', ('bob',))
('not_round', ('harry',))
('rough', ('gary',))
('young', ('erin',))


In [86]:
for i in range(100):
    example = dataset["dev"][i]

    if example["question"] == "Bob is nice.":
        print(example["context"])
        break

Bob is not red. Erin is young. Gary is rough. Harry is not round. If Bob is not round and Bob is not young then Bob is nice. All red people are not smart.


In [87]:
print(parse_atom("Bob is not round"))
print(parse_atom("Bob is not young"))
print(parse_atom("Bob is not red"))

('not_round', ('bob',))
('not_young', ('bob',))
('not_red', ('bob',))


In [88]:

sentence = "If Bob is not round and Bob is not young then Bob is nice."

print(parse_rule(sentence))

([('not_round', ('bob',)), ('not_young', ('bob',))], ('nice', ('bob',)))


In [89]:
example = None

for i in range(100):
    if dataset["dev"][i]["question"] == "Bob is nice.":
        example = dataset["dev"][i]
        break

facts, rules, unsupported = parse_context(example["context"])

print("FACTS:")
for fact in sorted(facts):
    print(fact)

print("\nRULES:")
for rule in rules:
    print(rule)

print("\nFORWARD CHAINING:")
all_facts = forward_chaining(facts, rules)

print(("nice", ("bob",)) in all_facts)

FACTS:
('not_red', ('bob',))
('not_round', ('harry',))
('rough', ('gary',))
('young', ('erin',))

RULES:
([('not_round', ('bob',)), ('not_young', ('bob',))], ('nice', ('bob',)))
([('red', ('?x',))], ('not_smart', ('?x',)))

FORWARD CHAINING:
False


In [90]:
target_context = example["context"]

for item in dataset["dev"]:
    if item["context"] == target_context:
        print("Question:", item["question"])
        print("Label:", item["label"])
        print("-" * 40)

Question: Gary is rough.
Label: entailment
----------------------------------------
Question: Bob is red.
Label: not entailment
----------------------------------------
Question: Bob is nice.
Label: entailment
----------------------------------------
Question: Bob is not nice.
Label: not entailment
----------------------------------------
Question: Bob is not rough.
Label: entailment
----------------------------------------
Question: Harry is not red.
Label: entailment
----------------------------------------
Question: Gary is young.
Label: not entailment
----------------------------------------


In [91]:
print("Configuration:", example["config"])

print("\nSame-context examples:")
for item in dataset["dev"]:
    if item["context"] == example["context"]:
        print(item["question"], "=>", item["label"])

Configuration: depth-1

Same-context examples:
Gary is rough. => entailment
Bob is red. => not entailment
Bob is nice. => entailment
Bob is not nice. => not entailment
Bob is not rough. => entailment
Harry is not red. => entailment
Gary is young. => not entailment


In [92]:
import inspect

print("FORWARD CHAINING:")
print(inspect.getsource(forward_chaining))

print("\nMATCH CONDITION:")
print(inspect.getsource(match_condition))

FORWARD CHAINING:
def forward_chaining(facts, rules):
    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:
            substitutions = [{}]

            for condition in conditions:
                new_substitutions = []

                for sub in substitutions:
                    new_substitutions.extend(
                        match_condition(condition, known_facts, sub)
                    )

                substitutions = new_substitutions

                if not substitutions:
                    break

            for sub in substitutions:
                predicate, arguments = conclusion

                derived = (
                    predicate,
                    tuple(
                        sub[arg] if arg.startswith("?") else arg
                        for arg in arguments
                    )
                )

                if derived not in known_facts:
                    know

In [93]:
context = """
Bob is not red.
Erin is young.
Gary is rough.
Harry is not round.
If Bob is not round and Bob is not young then Bob is nice.
All red people are not smart.
"""

facts, rules, unsupported = parse_context(context)
all_facts = forward_chaining(facts, rules)

print("Derived facts:")
for fact in sorted(all_facts):
    print(fact)

print("\nBob is nice:", answer_query(parse_atom("Bob is nice"), all_facts))
print("Bob is not nice:", answer_query(parse_atom("Bob is not nice"), all_facts))

Derived facts:
('not_red', ('bob',))
('not_round', ('harry',))
('rough', ('gary',))
('young', ('erin',))

Bob is nice: not entailment
Bob is not nice: entailment


In [94]:
for error in errors:
    print("Question:", error["question"])
    print("Actual:", error["actual"])
    print("Predicted:", error["predicted"])

    facts, rules, unsupported = parse_context(error["context"])

    print("Unsupported:", unsupported)
    print("Parsed rules:", rules)
    print("-" * 70)

Question: Bob is nice.
Actual: entailment
Predicted: not entailment
Unsupported: []
Parsed rules: [([('not_round', ('bob',)), ('not_young', ('bob',))], ('nice', ('bob',))), ([('red', ('?x',))], ('not_smart', ('?x',)))]
----------------------------------------------------------------------
Question: Bob is not nice.
Actual: not entailment
Predicted: entailment
Unsupported: []
Parsed rules: [([('not_round', ('bob',)), ('not_young', ('bob',))], ('nice', ('bob',))), ([('red', ('?x',))], ('not_smart', ('?x',)))]
----------------------------------------------------------------------
Question: Charlie is big.
Actual: entailment
Predicted: not entailment
Unsupported: []
Parsed rules: [([('round', ('?x',)), ('not_rough', ('?x',))], ('furry', ('?x',))), ([('round', ('?x',)), ('not_smart', ('?x',))], ('not_rough', ('?x',))), ([('blue', ('?x',)), ('not_rough', ('?x',))], ('big', ('?x',))), ([('smart', ('?x',)), ('blue', ('?x',))], ('not_round', ('?x',))), ([('round', ('?x',)), ('furry', ('?x',))],

In [95]:
for error in errors:
    if error["question"] == "Charlie is big.":
        facts, rules, unsupported = parse_context(error["context"])

        print("Facts:", facts)
        print("\nParsed rules:")
        for rule in rules:
            print(rule)

        print("\nUnsupported:", unsupported)
        break

Facts: {('blue', ('charlie',)), ('big', ('dave',)), ('smart', ('harry',)), ('big', ('gary',))}

Parsed rules:
([('round', ('?x',)), ('not_rough', ('?x',))], ('furry', ('?x',)))
([('round', ('?x',)), ('not_smart', ('?x',))], ('not_rough', ('?x',)))
([('blue', ('?x',)), ('not_rough', ('?x',))], ('big', ('?x',)))
([('smart', ('?x',)), ('blue', ('?x',))], ('not_round', ('?x',)))
([('round', ('?x',)), ('furry', ('?x',))], ('blue', ('?x',)))
([('rough', ('charlie',))], ('big', ('charlie',)))
([('furry', ('?x',)), ('smart', ('?x',))], ('big', ('?x',)))
([('rough', ('?x',))], ('not_furry', ('?x',)))

Unsupported: []


In [96]:
for error in errors:
    if error["question"] == "Bob is nice.":
        print("Original context:")
        print(error["context"])

        facts, rules, unsupported = parse_context(error["context"])
        all_facts = forward_chaining(facts, rules)

        print("\nFacts:", facts)
        print("\nDerived facts:", all_facts)
        break

Original context:
Bob is not red. Erin is young. Gary is rough. Harry is not round. If Bob is not round and Bob is not young then Bob is nice. All red people are not smart.

Facts: {('young', ('erin',)), ('rough', ('gary',)), ('not_round', ('harry',)), ('not_red', ('bob',))}

Derived facts: {('young', ('erin',)), ('rough', ('gary',)), ('not_round', ('harry',)), ('not_red', ('bob',))}


In [97]:
context = """
Bob is not red.
Erin is young.
Gary is rough.
Harry is not round.
If Bob is not round and Bob is not young then Bob is nice.
All red people are not smart.
"""

facts, rules, unsupported = parse_context(context)
all_facts = forward_chaining(facts, rules)

print("Derived facts:", all_facts)
print("Bob is nice:", answer_query(parse_atom("Bob is nice"), all_facts))
print("Bob is not nice:", answer_query(parse_atom("Bob is not nice"), all_facts))

Derived facts: {('young', ('erin',)), ('rough', ('gary',)), ('not_round', ('harry',)), ('not_red', ('bob',))}
Bob is nice: not entailment
Bob is not nice: entailment


In [98]:
for error in errors:
    if error["question"] == "Charlie is big.":
        facts, rules, unsupported = parse_context(error["context"])

        all_facts = forward_chaining(facts, rules)

        print("Question:", error["question"])
        print("Actual:", error["actual"])
        print("Predicted:", answer_query(parse_atom(error["question"]), all_facts))

        print("\nIs Charlie big?",
              ("big", ("charlie",)) in all_facts)

        print("\nUnsupported:", unsupported)
        break

Question: Charlie is big.
Actual: entailment
Predicted: not entailment

Is Charlie big? False

Unsupported: []


In [37]:
import inspect
print(inspect.getsource(parse_atom))

def parse_atom(sentence):
    sentence = sentence.strip().rstrip(".")
    

    sentence = re.sub(
        r"\b(someone|something|they|it)\b",
        "?x", sentence, flags=re.I
    )

    # Remove "the" at the beginning
    sentence = re.sub(r"^the\s+", "", sentence, flags=re.I)

    # Negative property
    m = re.fullmatch(r"(.+?)\s+(?:is|are)\s+not\s+(\w+)", sentence, re.I)
    if m:
        entity, prop = m.groups()
        return ("not_" + prop.lower(), (entity.lower(),))

    # Positive property
    m = re.fullmatch(r"(.+?)\s+(?:is|are)\s+(\w+)", sentence, re.I)
    if m:
        entity, prop = m.groups()
        return (prop.lower(), (entity.lower(),))

    # Known verbs
    verbs = [
        "chase", "chases",
        "see", "sees",
        "like", "likes",
        "need", "needs",
        "have", "has",
        "eat", "eats",
        "visit", "visits",
        "help", "helps"
    ]

    verb_pattern = "|".join(sorted(verbs, key=len, reverse=True))

    # Negative relationship


In [99]:
tests = [
    "The bald eagle is cold.",
    "The bald eagle chases the dog.",
    "The tiger sees the bald eagle.",
    "The tiger does not see the bald eagle.",
    "Bob is not round.",
    "Someone is blue."
]

for sentence in tests:
    print(sentence, "=>", parse_atom(sentence))

The bald eagle is cold. => ('cold', ('bald eagle',))
The bald eagle chases the dog. => ('chases', ('bald eagle', 'dog'))
The tiger sees the bald eagle. => ('sees', ('tiger', 'bald eagle'))
The tiger does not see the bald eagle. => ('not_sees', ('tiger', 'bald eagle'))
Bob is not round. => ('not_round', ('bob',))
Someone is blue. => ('blue', ('?x',))


In [100]:
for error in errors:
    if error["question"] == "The bald eagle is cold.":
        facts, rules, unsupported = parse_context(error["context"])

        all_facts = forward_chaining(facts, rules)

        print("Question:", error["question"])
        print("Actual:", error["actual"])
        print("Predicted:", answer_query(parse_atom(error["question"]), all_facts))

        print("\nBald eagle is cold:",
              ("cold", ("bald eagle",)) in all_facts)

        print("\nUnsupported:", unsupported)
        break

In [101]:
for error in errors:
    print("Question:", error["question"])
    print("Actual:", error["actual"])
    print("Predicted:", error["predicted"])

    facts, rules, unsupported = parse_context(error["context"])

    print("Unsupported:", unsupported)
    print("-" * 60)

Question: Bob is nice.
Actual: entailment
Predicted: not entailment
Unsupported: []
------------------------------------------------------------
Question: Bob is not nice.
Actual: not entailment
Predicted: entailment
Unsupported: []
------------------------------------------------------------
Question: Charlie is big.
Actual: entailment
Predicted: not entailment
Unsupported: []
------------------------------------------------------------
Question: Charlie is not big.
Actual: not entailment
Predicted: entailment
Unsupported: []
------------------------------------------------------------


In [102]:
for error in errors:

    if error["question"] in ["Bob is nice.", "Anne is big."]:

        print("\nQUESTION:", error["question"])
        print("\nCONTEXT:")
        print(error["context"])

        facts, rules, unsupported = parse_context(error["context"])
        all_facts = forward_chaining(facts, rules)

        print("\nEXTRACTED FACTS:")
        for fact in facts:
            print(fact)

        print("\nPARSED RULES:")
        for rule in rules:
            print(rule)

        print("\nDERIVED FACTS:")
        for fact in all_facts:
            print(fact)

        print("\nUNSUPPORTED:", unsupported)
        print("=" * 70)


QUESTION: Bob is nice.

CONTEXT:
Bob is not red. Erin is young. Gary is rough. Harry is not round. If Bob is not round and Bob is not young then Bob is nice. All red people are not smart.

EXTRACTED FACTS:
('young', ('erin',))
('rough', ('gary',))
('not_round', ('harry',))
('not_red', ('bob',))

PARSED RULES:
([('not_round', ('bob',)), ('not_young', ('bob',))], ('nice', ('bob',)))
([('red', ('?x',))], ('not_smart', ('?x',)))

DERIVED FACTS:
('young', ('erin',))
('rough', ('gary',))
('not_round', ('harry',))
('not_red', ('bob',))

UNSUPPORTED: []


In [103]:
for error in errors:
    if error["question"] in ["Bob is nice.", "Anne is big."]:

        facts, rules, unsupported = parse_context(error["context"])
        all_facts = forward_chaining(facts, rules)

        print("\nQuestion:", error["question"])
        print("Actual:", error["actual"])
        print("Predicted:", answer_query(
            parse_atom(error["question"]), all_facts
        ))
        print("Derived facts:", all_facts)
        print("Unsupported:", unsupported)
        print("-" * 50)


Question: Bob is nice.
Actual: entailment
Predicted: not entailment
Derived facts: {('young', ('erin',)), ('rough', ('gary',)), ('not_round', ('harry',)), ('not_red', ('bob',))}
Unsupported: []
--------------------------------------------------


In [104]:
print(parse_rule(
    "If something is not smart then it is nice"
))

print(parse_atom("something is not smart"))
print(parse_atom("it is nice"))

([('not_smart', ('?x',))], ('nice', ('?x',)))
('not_smart', ('?x',))
('nice', ('?x',))


In [105]:
context = """
Bob is not smart.
If something is not smart then it is nice.
"""

facts, rules, unsupported = parse_context(context)

print("Facts:", facts)
print("Rules:", rules)

all_facts = forward_chaining(facts, rules)

print("Derived facts:", all_facts)

Facts: {('not_smart', ('bob',))}
Rules: [([('not_smart', ('?x',))], ('nice', ('?x',)))]
Derived facts: {('not_smart', ('bob',)), ('nice', ('bob',))}


In [106]:

# COMPLETE RULETAKER TESTING

# Test 1: Basic parser tests
print("=" * 60)
print("TEST 1: PARSER TESTS")
print("=" * 60)

test_sentences = [
    "Bob is not smart.",
    "Anne is big.",
    "The bald eagle is cold.",
    "The bald eagle chases the dog.",
    "The tiger sees the bald eagle.",
    "The tiger does not see the bald eagle.",
    "Someone is blue."
]

for sentence in test_sentences:
    print(sentence, "=>", parse_atom(sentence))


# Test 2: Rule parsing tests
print("\n" + "=" * 60)
print("TEST 2: RULE PARSING")
print("=" * 60)

test_rules = [
    "All blue things are white.",
    "All quiet things are not young.",
    "If something is not smart then it is nice.",
    "If someone is round and not rough then they are furry."
]

for sentence in test_rules:
    print(sentence, "=>", parse_rule(sentence))


# Test 3: Bob reasoning
print("\n" + "=" * 60)
print("TEST 3: BOB REASONING")
print("=" * 60)

bob_context = """
Bob is not smart.
If something is not smart then it is nice.
"""

facts, rules, unsupported = parse_context(bob_context)
bob_facts = forward_chaining(facts, rules)

print("Derived facts:", bob_facts)
print("Bob is nice:", answer_query(parse_atom("Bob is nice."), bob_facts))


# Test 4: Anne reasoning
print("\n" + "=" * 60)
print("TEST 4: ANNE REASONING")
print("=" * 60)

anne_context = """
Anne is not nice.
If something is not nice then it is big.
"""

facts, rules, unsupported = parse_context(anne_context)
anne_facts = forward_chaining(facts, rules)

print("Derived facts:", anne_facts)
print("Anne is big:", answer_query(parse_atom("Anne is big."), anne_facts))


# Test 5: Development dataset evaluation
print("\n" + "=" * 60)
print("TEST 5: RULETAKER DEV EVALUATION")
print("=" * 60)

total = 100
correct = 0
errors = []

for i in range(total):
    example = dataset["dev"][i]

    facts, rules, unsupported = parse_context(example["context"])
    all_facts = forward_chaining(facts, rules)

    query = parse_atom(example["question"])
    prediction = answer_query(query, all_facts)

    actual = example["label"]

    if prediction == actual:
        correct += 1
    else:
        errors.append({
            "question": example["question"],
            "actual": actual,
            "predicted": prediction
        })

print("Total examples:", total)
print("Correct:", correct)
print("Incorrect:", len(errors))
print("Accuracy:", correct / total * 100, "%")

print("\nRemaining errors:")
for error in errors[:10]:
    print(error)


TEST 1: PARSER TESTS
Bob is not smart. => ('not_smart', ('bob',))
Anne is big. => ('big', ('anne',))
The bald eagle is cold. => ('cold', ('bald eagle',))
The bald eagle chases the dog. => ('chases', ('bald eagle', 'dog'))
The tiger sees the bald eagle. => ('sees', ('tiger', 'bald eagle'))
The tiger does not see the bald eagle. => ('not_sees', ('tiger', 'bald eagle'))
Someone is blue. => ('blue', ('?x',))

TEST 2: RULE PARSING
All blue things are white. => None
All quiet things are not young. => None
If something is not smart then it is nice. => ([('not_smart', ('?x',))], ('nice', ('?x',)))
If someone is round and not rough then they are furry. => ([('round', ('?x',)), ('not_rough', ('?x',))], ('furry', ('?x',)))

TEST 3: BOB REASONING
Derived facts: {('not_smart', ('bob',)), ('nice', ('bob',))}
Bob is nice: entailment

TEST 4: ANNE REASONING
Derived facts: {('not_nice', ('anne',)), ('big', ('anne',))}
Anne is big: entailment

TEST 5: RULETAKER DEV EVALUATION
Total examples: 100
Correct

In [110]:
for i in range(100):
    example = dataset["dev"][i]

    if example["question"] in ["Bob is nice.", "Charlie is big."]:

        facts, rules, unsupported = parse_context(example["context"])
        all_facts = forward_chaining(facts, rules)

        print("\nQuestion:", example["question"])
        print("Actual:", example["label"])

        print("Predicted:", answer_query(
            parse_atom(example["question"]), all_facts
        ))

        print("Derived facts:", all_facts)
        print("Unsupported:", unsupported)
        print("-" * 60)


Question: Bob is nice.
Actual: entailment
Predicted: not entailment
Derived facts: {('young', ('erin',)), ('rough', ('gary',)), ('not_round', ('harry',)), ('not_red', ('bob',))}
Unsupported: ['All red people are not smart']
------------------------------------------------------------

Question: Bob is nice.
Actual: entailment
Predicted: entailment
Derived facts: {('not_smart', ('bob',)), ('nice', ('bob',))}
Unsupported: []
------------------------------------------------------------

Question: Charlie is big.
Actual: entailment
Predicted: not entailment
Derived facts: {('blue', ('charlie',)), ('big', ('dave',)), ('smart', ('harry',)), ('big', ('gary',))}
Unsupported: ['If someone is round and not smart then they are not rough', 'If someone is smart and blue then they are not round', 'If someone is rough then they are not furry']
------------------------------------------------------------


In [112]:
print(parse_rule("All red people are not smart."))
print(parse_rule("All blue people are green."))

([('red', ('?x',))], ('not_smart', ('?x',)))
([('blue', ('?x',))], ('green', ('?x',)))


In [113]:
tests = [
    "If someone is round and not smart then they are not rough.",
    "If someone is smart and blue then they are not round.",
    "If someone is rough then they are not furry."
]

for sentence in tests:
    print("\n", sentence)
    print(parse_rule(sentence))


 If someone is round and not smart then they are not rough.
([('round', ('?x',)), ('not_smart', ('?x',))], ('not_rough', ('?x',)))

 If someone is smart and blue then they are not round.
([('smart', ('?x',)), ('blue', ('?x',))], ('not_round', ('?x',)))

 If someone is rough then they are not furry.
([('rough', ('?x',))], ('not_furry', ('?x',)))


In [115]:
# Find Charlie's original example in dev dataset

for i in range(100):
    ex = dataset["dev"][i]

    if ex["question"] == "Charlie is big.":

        facts, rules, unsupported = parse_context(ex["context"])
        derived = forward_chaining(facts, rules)

        print("ORIGINAL CONTEXT:")
        print(ex["context"])

        print("\nNUMBER OF FACTS:", len(facts))
        print("NUMBER OF RULES:", len(rules))

        print("\nPARSED RULES:")
        for r in rules:
            print(r)

        print("\nIS CHARLIE BIG?",
              ("big", ("charlie",)) in derived)

        print("\nUNSUPPORTED:", unsupported)

        break

ORIGINAL CONTEXT:
Charlie is blue. Dave is big. Gary is big. Harry is smart. If someone is round and not rough then they are furry. If someone is round and not smart then they are not rough. If someone is blue and not rough then they are big. If someone is smart and blue then they are not round. If someone is round and furry then they are blue. If Charlie is rough then Charlie is big. Furry, smart people are big. If someone is rough then they are not furry.

NUMBER OF FACTS: 4
NUMBER OF RULES: 8

PARSED RULES:
([('round', ('?x',)), ('not_rough', ('?x',))], ('furry', ('?x',)))
([('round', ('?x',)), ('not_smart', ('?x',))], ('not_rough', ('?x',)))
([('blue', ('?x',)), ('not_rough', ('?x',))], ('big', ('?x',)))
([('smart', ('?x',)), ('blue', ('?x',))], ('not_round', ('?x',)))
([('round', ('?x',)), ('furry', ('?x',))], ('blue', ('?x',)))
([('rough', ('charlie',))], ('big', ('charlie',)))
([('furry', ('?x',)), ('smart', ('?x',))], ('big', ('?x',)))
([('rough', ('?x',))], ('not_furry', ('?x'

In [116]:
facts, rules, unsupported = parse_context(ex["context"])

print("Initial facts:", facts)

print("\nTest matching round:")
print(match_condition(("round", ("?x",)), facts, {}))

print("\nTest matching not_smart:")
print(match_condition(("not_smart", ("?x",)), facts, {}))

print("\nTest matching not_rough:")
print(match_condition(
    ("not_rough", ("?x",)),
    facts,
    {"?x": "charlie"}
))

Initial facts: {('blue', ('charlie',)), ('big', ('dave',)), ('smart', ('harry',)), ('big', ('gary',))}

Test matching round:
[]

Test matching not_smart:
[]

Test matching not_rough:
[]


In [117]:
import inspect

print("MATCH CONDITION:")
print(inspect.getsource(match_condition))

print("\nFORWARD CHAINING:")
print(inspect.getsource(forward_chaining))

print("\nANSWER QUERY:")
print(inspect.getsource(answer_query))

MATCH CONDITION:
def match_condition(condition, facts, substitution):
    predicate, arguments = condition
    matches = []

    for fact_predicate, fact_arguments in facts:
        if predicate != fact_predicate:
            continue

        if len(arguments) != len(fact_arguments):
            continue

        updated = substitution.copy()
        valid = True

        for pattern, value in zip(arguments, fact_arguments):
            if pattern.startswith("?"):
                if pattern in updated and updated[pattern] != value:
                    valid = False
                    break
                updated[pattern] = value
            elif pattern != value:
                valid = False
                break

        if valid:
            matches.append(updated)

    return matches


FORWARD CHAINING:
def forward_chaining(facts, rules):
    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:
    

In [122]:
total = 100
correct = 0
errors = []

for i in range(total):
    example = dataset["dev"][i]

    facts, rules, unsupported = parse_context(example["context"])
    all_facts = forward_chaining(facts, rules)

    query = parse_atom(example["question"])
    prediction = answer_query(query, all_facts)

    actual = example["label"]

    if prediction == actual:
        correct += 1
    else:
        errors.append({
            "question": example["question"],
            "actual": actual,
            "predicted": prediction,
            "context": example["context"]
        })

print("Total examples:", total)
print("Correct:", correct)
print("Incorrect:", len(errors))
print("Accuracy:", correct / total * 100, "%")

print("\nRemaining errors:")
for error in errors[:10]:
    print(error["question"])
    print("Actual:", error["actual"])
    print("Predicted:", error["predicted"])
    print("-" * 40)

Total examples: 100
Correct: 100
Incorrect: 0
Accuracy: 100.0 %

Remaining errors:


In [164]:
total = 1000
correct = 0

for i in range(total):
    example = dataset["dev"][i]

    facts, rules, unsupported = parse_context(example["context"])
    all_facts = forward_chaining(facts, rules)

    query = parse_atom(example["question"])
    prediction = answer_query(query, all_facts)

    if prediction == example["label"]:
        correct += 1

print("Total examples:", total)
print("Correct predictions:", correct)
print("Accuracy:", correct / total * 100, "%")

Total examples: 1000
Correct predictions: 897
Accuracy: 89.7 %


In [124]:
print(forward_chaining)

<function forward_chaining at 0x786124613560>


In [125]:
import inspect

print(inspect.getsource(forward_chaining))

def forward_chaining(facts, rules):

    known_facts = set(facts)
    changed = True

    while changed:
        changed = False

        for conditions, conclusion in rules:

            substitutions = [{}]

            # Match all conditions of the rule
            for condition in conditions:

                new_substitutions = []

                for substitution in substitutions:
                    matches = match_condition(
                        condition,
                        known_facts,
                        substitution
                    )
                    new_substitutions.extend(matches)

                substitutions = new_substitutions

                if not substitutions:
                    break

            # Derive the conclusion
            for substitution in substitutions:

                predicate, arguments = conclusion

                derived_arguments = tuple(
                    substitution[arg] if arg.startswith("?") else arg
             

In [126]:

example = dataset["dev"][0]

print("Dataset keys:", example.keys())
print("Context:", example["context"])
print("Question:", example["question"])
print("Label:", example["label"])

Dataset keys: dict_keys(['context', 'question', 'label', 'config'])
Context: The cat chases the cow. The cat is big. The cat is red. The cat is round. The cat is young. The cat needs the tiger. The cow chases the lion. The cow likes the tiger. The cow needs the cat. The lion chases the tiger. The lion is young. The tiger needs the lion. If someone likes the cat then they are young. If the cow chases the lion then the lion needs the tiger. If someone chases the cat then they are cold. If someone is young then they chase the lion. If someone likes the cow and the cow chases the lion then the cow chases the tiger. If someone likes the lion and they chase the lion then the lion is cold. If someone likes the tiger and the tiger needs the cow then they chase the cow. If someone needs the cat and the cat likes the tiger then they are young.
Question: The cow likes the tiger.
Label: entailment


In [127]:
example = dataset["dev"][0]

facts, rules, unsupported = parse_context(example["context"])

print("Number of facts:", len(facts))
print("Number of rules:", len(rules))
print("Unsupported sentences:", unsupported)

all_facts = forward_chaining(facts, rules)

print("Total derived facts:", len(all_facts))

Number of facts: 12
Number of rules: 8
Unsupported sentences: []
Total derived facts: 15


In [139]:
total = 1000
correct = 0

for i in range(total):
    example = dataset["dev"][i]

    facts, rules, unsupported = parse_context(example["context"])
    all_facts = forward_chaining(facts, rules)

    query = parse_atom(example["question"])
    prediction = answer_query(query, all_facts)

    if prediction == example["label"]:
        correct += 1

print("Total:", total)
print("Correct:", correct)
print("Accuracy:", correct / total * 100, "%")

Total: 1000
Correct: 897
Accuracy: 89.7 %


In [132]:
for i in range(1000):
    example = dataset["dev"][i]
    facts, rules, unsupported = parse_context(example["context"])

    try:
        forward_chaining(facts, rules)
    except KeyError as e:
        print("Error at example:", i)
        print("Missing variable:", e)
        print("Context:", example["context"])
        print("Rules:", rules)
        break

Error at example: 342
Missing variable: '?x do not'
Context: The bald eagle is red. The bald eagle does not like the bear. The bald eagle does not like the squirrel. The bald eagle needs the squirrel. The bear does not like the bald eagle. The bear needs the bald eagle. The bear needs the squirrel. The bear visits the squirrel. The squirrel likes the bear. The squirrel does not need the bald eagle. The squirrel needs the bear. The squirrel visits the bald eagle. If the squirrel needs the bald eagle and the squirrel is young then the bald eagle likes the bear. If someone visits the bald eagle and the bald eagle is red then they do not like the bald eagle.
Rules: [([('needs', ('squirrel', 'bald eagle')), ('young', ('squirrel',))], ('likes', ('bald eagle', 'bear'))), ([('visits', ('?x', 'bald eagle')), ('red', ('bald eagle',))], ('likes', ('?x do not', 'bald eagle')))]


In [138]:
example = dataset["dev"][342]

facts, rules, unsupported = parse_context(example["context"])

print("Unsupported:", unsupported)
print("Parsed rules:", rules)

Unsupported: []
Parsed rules: [([('needs', ('squirrel', 'bald eagle')), ('young', ('squirrel',))], ('likes', ('bald eagle', 'bear'))), ([('visits', ('?x', 'bald eagle')), ('red', ('bald eagle',))], ('not_likes', ('?x', 'bald eagle')))]


In [141]:
from datasets import load_dataset

folio = load_dataset("tasksource/folio")

print(folio)
print(folio["train"][0])

DatasetDict({
    train: Dataset({
        features: ['story_id', 'premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label', 'example_id'],
        num_rows: 1001
    })
    validation: Dataset({
        features: ['story_id', 'premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label', 'example_id'],
        num_rows: 203
    })
})
{'story_id': 406, 'premises': "All people who regularly drink coffee are dependent on caffeine.\nPeople regularly drink coffee, or they don't want to be addicted to caffeine, or both.\nNo one who doesn't want to be addicted to caffeine is unaware that caffeine is a drug.\nRina is either a student who is unaware that caffeine is a drug, or she is not a student and is she aware that caffeine is a drug.\nRina  is either a student who is dependent on caffeine, or she is not a student and not dependent on caffeine.", 'premises-FOL': '∀x (DrinkRegularly(x, coffee) → IsDependentOn(x, caffeine))\n∀x (DrinkRegularly(x, coffee)  ∨ (¬WantToBeAddicted

In [142]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(
    repo_id="counterbench/CounterBench",
    repo_type="dataset"
)

print(files)

['.gitattributes', 'README.md', 'data_balanced_alpha_V1.json', 'data_balanced_backdoor_V2.json', 'meta_model_alpha_V1.json', 'meta_model_backdoor_V2.json']


In [143]:
import pandas as pd

url = "https://huggingface.co/datasets/counterbench/CounterBench/resolve/main/data_balanced_alpha_V1.json"

counterbench = pd.read_json(url)

print(counterbench.shape)
print(counterbench.head(2))
print(counterbench.columns.tolist())

(1000, 6)
                                           question  \
0  Would Lumbo occur if not Ziklo instead of Ziklo?   
1   Would Zlim occur if not Glent instead of Glent?   

                                          given_info answer   type  \
0  We know that Ziklo causes Blaf, Blaf causes Tr...     no  basic   
1  We know that Glent causes Razz, Razz causes Pe...     no  basic   

   question_id                                               meta  
0            0  {'story_id': 'nonsense_0', 'rung': 3, 'query_t...  
1          200  {'story_id': 'nonsense_1', 'rung': 3, 'query_t...  
['question', 'given_info', 'answer', 'type', 'question_id', 'meta']


In [144]:
import time

total = 1000
correct = 0

start = time.time()

for i in range(total):
    example = dataset["dev"][i]

    facts, rules, unsupported = parse_context(example["context"])
    all_facts = forward_chaining(facts, rules)

    query = parse_question(example["question"])
    prediction = answer_query(query, all_facts)

    if prediction == example["label"]:
        correct += 1

end = time.time()

print("Dataset: RuleTaker")
print("Total:", total)
print("Accuracy:", correct / total * 100, "%")
print("Execution time:", round(end - start, 3), "seconds")

Dataset: RuleTaker
Total: 1000
Accuracy: 89.7 %
Execution time: 0.439 seconds


In [145]:
print("========== FOLIO ==========")

for i in range(3):
    print("\nExample:", i)
    print("Premises:", folio["validation"][i]["premises"])
    print("Conclusion:", folio["validation"][i]["conclusion"])
    print("Label:", folio["validation"][i]["label"])


print("\n========== COUNTERBENCH ==========")

for i in range(5):
    print("\nExample:", i)
    print("Given info:", counterbench.iloc[i]["given_info"])
    print("Question:", counterbench.iloc[i]["question"])
    print("Answer:", counterbench.iloc[i]["answer"])

========== FOLIO ==========

Example: 0
Premises: People in this club who perform in school talent shows often attend and are very engaged with school events.
People in this club either perform in school talent shows often or are inactive and disinterested community members.
People in this club who chaperone high school dances are not students who attend the school.
All people in this club who are inactive and disinterested members of their community chaperone high school dances.
All young children and teenagers in this club who wish to further their academic careers and educational opportunities are students who attend the school. 
Bonnie is in this club and she either both attends and is very engaged with school events and is a student who attends the school or is not someone who both attends and is very engaged with school events and is not a student who attends the school.
Conclusion: Bonnie performs in school talent shows often.
Label: Uncertain

Example: 1
Premises: People in thi

In [150]:
import re
import time

import re

def solve_counterbench(info, question):

    # Extract facts that are explicitly observed
    observed = re.findall(r"We observed (\w+)", info, re.I)
    observed += re.findall(r"we observed (\w+)", info, re.I)

    # Extract counterfactual assumptions
    assumptions = re.findall(r"not (\w+)", question, re.I)

    # Remove the original causes under intervention
    facts = set(observed)

    # Extract causal rules
    text = re.sub(r"^We know that\s+", "", info.strip(), flags=re.I)
    text = re.sub(r"\.\s*We observed.*$", "", text, flags=re.I)

    rules = []

    # Split individual statements
    statements = re.split(r",\s*|\s+and\s+(?=[A-Z])", text)

    for statement in statements:
        m = re.fullmatch(
            r"(.+?)\s+causes\s+(.+)",
            statement.strip(),
            re.I
        )

        if not m:
            continue

        causes, effect = m.groups()

        conditions = re.split(r"\s+and\s+|\s+or\s+", causes, flags=re.I)
        effect = effect.strip().rstrip(".")

        rules.append((conditions, effect))

    # Forward chaining
    changed = True

    while changed:
        changed = False

        for conditions, effect in rules:
            if any(c in assumptions for c in conditions):
                continue

            if all(c in facts for c in conditions):
                if effect not in facts:
                    facts.add(effect)
                    changed = True

    target_match = re.search(r"Would\s+(\w+)", question, re.I)

    if target_match:
        target = target_match.group(1)
        return "yes" if target in facts else "no"

    return None

In [147]:
correct = 0
start = time.time()

for _, row in counterbench.iterrows():
    prediction = solve_counterbench(
        row["given_info"],
        row["question"]
    )

    if prediction == row["answer"]:
        correct += 1

runtime = time.time() - start

print("CounterBench examples:", len(counterbench))
print("Correct:", correct)
print("Accuracy:", round(correct / len(counterbench) * 100, 2), "%")
print("Runtime:", round(runtime, 3), "seconds")

CounterBench examples: 1000
Correct: 400
Accuracy: 40.0 %
Runtime: 0.144 seconds


In [148]:
print(counterbench["type"].value_counts())
print("\nAnswer distribution:")
print(counterbench["answer"].value_counts())

print("\nExamples:")
print(counterbench[["question", "given_info", "answer", "type"]].head(10).to_string(index=False))


type
basic          250
conditional    250
joint          250
nested         250
Name: count, dtype: int64

Answer distribution:
answer
no     500
yes    500
Name: count, dtype: int64

Examples:
                                        question                                                                                                         given_info answer  type
Would Lumbo occur if not Ziklo instead of Ziklo?                       We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.     no basic
 Would Zlim occur if not Glent instead of Glent?                            We know that Glent causes Razz, Razz causes Pex, Pex causes Zurn, and Zurn causes Zlim.     no basic
   Would Klep occur if not Praf instead of Praf?                         We know that Praf causes Vank, Vank causes Scud, Scud causes Wrenk, and Wrenk causes Klep.     no basic
   Would Jext occur if not Fizo instead of Fizo?                         We know that Fizo causes

In [149]:
for t in ["basic", "conditional", "joint", "nested"]:
    print("\n==========", t.upper(), "==========")

    samples = counterbench[counterbench["type"] == t].head(2)

    for _, row in samples.iterrows():
        print("Given:", row["given_info"])
        print("Question:", row["question"])
        print("Expected:", row["answer"])
        print()


========== BASIC ==========
Given: We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.
Question: Would Lumbo occur if not Ziklo instead of Ziklo?
Expected: no

Given: We know that Glent causes Razz, Razz causes Pex, Pex causes Zurn, and Zurn causes Zlim.
Question: Would Zlim occur if not Glent instead of Glent?
Expected: no


========== CONDITIONAL ==========
Given: We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune
Question: Would Lumbo occur if not Ziklo instead of Ziklo?
Expected: yes

Given: We know that Glent causes not Razz, Razz and Pex together cause Zurn, Zurn causes Zlim. We observed Pex
Question: Would Zlim occur if not Glent instead of Glent?
Expected: yes


========== JOINT ==========
Given: We know that Ziklo causes Blaf and Vork, Blaf causes not Trune, Trune or Vork causes Lumbo.
Question: Would Lumbo occur if not Ziklo and not Blaf?
Expected: yes

Given: We know that Glen

In [151]:
for t in ["basic", "conditional", "joint", "nested"]:
    row = counterbench[counterbench["type"] == t].iloc[0]

    print(t)
    print("Expected:", row["answer"])
    print("Predicted:", solve_counterbench(
        row["given_info"], row["question"]
    ))
    print()

basic
Expected: no
Predicted: no

conditional
Expected: yes
Predicted: no

joint
Expected: yes
Predicted: no

nested
Expected: yes
Predicted: no



In [152]:
row = counterbench[counterbench["type"] == "conditional"].iloc[0]

print(row["given_info"])
print(row["question"])

We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune
Would Lumbo occur if not Ziklo instead of Ziklo?


In [153]:
from huggingface_hub import hf_hub_download

readme_path = hf_hub_download(
    repo_id="counterbench/CounterBench",
    filename="README.md",
    repo_type="dataset"
)

with open(readme_path, "r") as f:
    print(f.read())

---
license: mit
---
# CounterBench-Deterministic-Counterfactual-1K-Dataset

## Dataset Description

CounterBench comprises 1.2K counterfactual reasoning questions. It can be found in ``data_balanced_alpha_V1.json'' and ``data_balanced_backdoor_V2.json''
It is designed with varying levels of difficulty, diverse causal graph structures, distinct types of counterfactual questions, and multiple nonsensical name variants. 

This dataset consists of two main components: 
a set of counterfactual queries and corresponding binary answers. These questions and answers are based on the following important columns:
- `question_id`: It represents a unique identifier for each question within the file. 
- `given_info`: It contains supplementary background or contextual information, expressed in natural language, that the model may need in order to provide a more accurate and well-informed answer to the question.
- `question`: It stores the specific question being asked, also formulated in natural lan

In [154]:
url2 = "https://huggingface.co/datasets/counterbench/CounterBench/resolve/main/data_balanced_backdoor_V2.json"

counterbench_v2 = pd.read_json(url2)

print(counterbench_v2.shape)
print(counterbench_v2["type"].value_counts())
print(counterbench_v2.head(3).to_string(index=False))

(200, 6)
type
backdoor    200
Name: count, dtype: int64
                                        question                                                                                                                               given_info answer     type  question_id                                                                                                           meta
Would Lumbo occur if not Ziklo instead of Ziklo? We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune    yes backdoor            0 {'story_id': 'nonsense_0', 'rung': 3, 'query_type': 'det-counterfactual', 'model_id': 0, 'graph_id': 'graph5'}
 Would Zlim occur if not Glent instead of Glent?        We know that Razz causes Glent, Glent or Razz causes Pex, Pex causes Zurn, and Zurn causes Zlim. Razz~ Bern(0.2). We observed Pex    yes backdoor          200 {'story_id': 'nonsense_1', 'rung': 3, 'query_type': 'det-counterfactual', 'mo

In [155]:
correct = 0
start = time.time()

for _, row in counterbench_v2.iterrows():
    prediction = solve_counterbench(
        row["given_info"],
        row["question"]
    )

    if prediction == row["answer"]:
        correct += 1

print("Total:", len(counterbench_v2))
print("Correct:", correct)
print("Accuracy:", round(correct / len(counterbench_v2) * 100, 2), "%")
print("Runtime:", round(time.time() - start, 3), "seconds")

Total: 200
Correct: 105
Accuracy: 52.5 %
Runtime: 0.094 seconds


In [156]:
print("Premises FOL:")
print(folio["validation"][0]["premises-FOL"])

print("\nConclusion FOL:")
print(folio["validation"][0]["conclusion-FOL"])

print("\nLabel:")
print(folio["validation"][0]["label"])

Premises FOL:
∀x (InThisClub(x) ∧ PerformOftenIn(x, schoolTalentShow) → Attend(x, schoolEvent) ∧ VeryEngagedWith(x, schoolEvent))
∀x (InThisClub(x) → PerformOftenIn(x, schoolTalentShow) ⊕ (InActive(x) ∧ Disinterested(x) ∧ MemberOf(x, community)))
∀x (InThisClub(x) ∧ Chaperone(x, highSchoolDance) → ¬(Studen(x) ∧ AttendSchool(x)))
∀x (InThisClub(x) ∧ (InActive(x) ∧ Disinterested(x) ∧ MemberOf(x, community)) → Chaperone(x, highSchoolDances))
∀x (InThisClub(x) ∧ (YoungChildren(x) ⊕ Teenager(x)) ∧ WishToFurther(x, academicCareer)) → Studen(x) ∧ AttendSchool(x))
InThisClub(bonnie) ∧ ¬((Attend(x, schoolEvent) ∧ VeryEngagedWith(bonnie, schoolEvent)) ⊕ (Studen(bonne) ∧ AttendSchool(bonnie)))

Conclusion FOL:
InThisClub(bonnie) ∧ (Perform(bonnie, schoolTalentShow))

Label:
Uncertain


In [157]:
from collections import Counter

print(Counter(folio["validation"]["label"]))
print(Counter(folio["train"]["label"]))

Counter({'True': 72, 'Uncertain': 69, 'False': 62})
Counter({'True': 388, 'Uncertain': 324, 'False': 289})


In [158]:
!pip install -q sympy

In [159]:
import sympy
print("SymPy version:", sympy.__version__)

SymPy version: 1.14.0


In [160]:
from sympy import symbols
from sympy.logic.inference import satisfiable

P, Q, R = symbols('P Q R')

# Facts and rules
knowledge_base = P & (P >> Q) & (Q >> R)

# Check whether R is entailed
result = not satisfiable(knowledge_base & ~R)

print("Is R entailed?", result)

Is R entailed? True


In [161]:
from sympy import symbols
from sympy.logic.inference import satisfiable

def check_entailment(kb, query):
    result = satisfiable(kb & ~query)
    return result is False

P, Q, R = symbols('P Q R')

kb = P & (P >> Q) & (Q >> R)

print("R:", check_entailment(kb, R))
print("P:", check_entailment(kb, P))
print("Not R:", check_entailment(kb, ~R))

R: True
P: True
Not R: False


In [162]:
errors = []

for i in range(1000):
    example = dataset["dev"][i]

    facts, rules, unsupported = parse_context(example["context"])
    all_facts = forward_chaining(facts, rules)

    query = parse_question(example["question"])
    prediction = answer_query(query, all_facts)

    if prediction != example["label"]:
        errors.append({
            "question": example["question"],
            "actual": example["label"],
            "predicted": prediction,
            "context": example["context"]
        })

print("Total errors:", len(errors))

for e in errors[:5]:
    print("\nQuestion:", e["question"])
    print("Expected:", e["actual"])
    print("Predicted:", e["predicted"])

Total errors: 103

Question: Charlie is green.
Expected: entailment
Predicted: not entailment

Question: Charlie is not green.
Expected: not entailment
Predicted: entailment

Question: Bob is green.
Expected: entailment
Predicted: not entailment

Question: Bob is not green.
Expected: not entailment
Predicted: entailment

Question: Charlie is quiet.
Expected: entailment
Predicted: not entailment


In [165]:
for i in range(5):
    row = folio["validation"][i]

    print("\nExample:", i)
    print("Premises FOL:", row["premises-FOL"])
    print("Conclusion FOL:", row["conclusion-FOL"])
    print("Label:", row["label"])


Example: 0
Premises FOL: ∀x (InThisClub(x) ∧ PerformOftenIn(x, schoolTalentShow) → Attend(x, schoolEvent) ∧ VeryEngagedWith(x, schoolEvent))
∀x (InThisClub(x) → PerformOftenIn(x, schoolTalentShow) ⊕ (InActive(x) ∧ Disinterested(x) ∧ MemberOf(x, community)))
∀x (InThisClub(x) ∧ Chaperone(x, highSchoolDance) → ¬(Studen(x) ∧ AttendSchool(x)))
∀x (InThisClub(x) ∧ (InActive(x) ∧ Disinterested(x) ∧ MemberOf(x, community)) → Chaperone(x, highSchoolDances))
∀x (InThisClub(x) ∧ (YoungChildren(x) ⊕ Teenager(x)) ∧ WishToFurther(x, academicCareer)) → Studen(x) ∧ AttendSchool(x))
InThisClub(bonnie) ∧ ¬((Attend(x, schoolEvent) ∧ VeryEngagedWith(bonnie, schoolEvent)) ⊕ (Studen(bonne) ∧ AttendSchool(bonnie)))
Conclusion FOL: InThisClub(bonnie) ∧ (Perform(bonnie, schoolTalentShow))
Label: Uncertain

Example: 1
Premises FOL: ∀x (InThisClub(x) ∧ PerformOftenIn(x, schoolTalentShow) → Attend(x, schoolEvent) ∧ VeryEngagedWith(x, schoolEvent))
∀x (InThisClub(x) → PerformOftenIn(x, schoolTalentShow) ⊕ (InAct

In [166]:
import nltk
print("NLTK version:", nltk.__version__)

from nltk.sem.logic import LogicParser
from nltk.inference import ResolutionProver

print("First-order logic prover is available.")

NLTK version: 3.9.1
First-order logic prover is available.


In [167]:
from nltk.sem.logic import LogicParser
from nltk.inference import ResolutionProver

lp = LogicParser()

premises = [
    lp.parse("all x.(human(x) -> mortal(x))"),
    lp.parse("human(socrates)")
]

query = lp.parse("mortal(socrates)")

result = ResolutionProver().prove(query, premises)

print("Query: mortal(socrates)")
print("Result:", result)

Query: mortal(socrates)
Result: True


In [174]:
import re

def convert_fol(text):
    text = text.replace("∀", "all ")
    text = text.replace("∃", "exists ")
    text = text.replace("∧", "&")
    text = text.replace("∨", "|")
    text = text.replace("¬", "-")
    text = text.replace("→", "->")

    # Remove unmatched closing parentheses only
    balance = 0
    result = ""

    for char in text:
        if char == "(":
            balance += 1
            result += char
        elif char == ")":
            if balance > 0:
                balance -= 1
                result += char
        else:
            result += char

    return result

In [169]:
def xor_formula(A, B):
    return (A | B) & -(A & B)

A = lp.parse("human(socrates)")
B = lp.parse("animal(socrates)")

formula = xor_formula(A, B)

print("XOR formula:", formula)
print("Parsed successfully:", formula is not None)

XOR formula: ((human(socrates) | animal(socrates)) & -(human(socrates) & animal(socrates)))
Parsed successfully: True


In [170]:
premises = [
    lp.parse("((human(socrates) | animal(socrates)) & -(human(socrates) & animal(socrates)))"),
    lp.parse("human(socrates)")
]

query = lp.parse("-animal(socrates)")

result = ResolutionProver().prove(query, premises)

print("Expected: True")
print("Actual:", result)

Expected: True
Actual: True


In [173]:
parsed_count = 0
failed_count = 0

for i in range(5):
    row = folio["validation"][i]

    formulas = row["premises-FOL"].split("\n")
    formulas.append(row["conclusion-FOL"])

    for formula in formulas:
        try:
            converted = convert_fol(formula)
            lp.parse(converted)
            parsed_count += 1
        except Exception as e:
            failed_count += 1
            print(f"Example {i} parsing error:")
            print(formula)
            print(e)

print("\nSuccessfully parsed:", parsed_count)
print("Failed:", failed_count)

Example 0 parsing error:
∀x (InThisClub(x) → PerformOftenIn(x, schoolTalentShow) ⊕ (InActive(x) ∧ Disinterested(x) ∧ MemberOf(x, community)))
Unexpected token: '⊕'.  Expected token ')'.
all x (InThisClub(x) -> PerformOftenIn(x, schoolTalentShow) ⊕ (InActive(x) & Disinterested(x) & MemberOf(x, community)))
                                                            ^
Example 0 parsing error:
∀x (InThisClub(x) ∧ (YoungChildren(x) ⊕ Teenager(x)) ∧ WishToFurther(x, academicCareer)) → Studen(x) ∧ AttendSchool(x))
Unexpected token: '⊕'.  Expected token ')'.
all x (InThisClub(x) & (YoungChildren(x) ⊕ Teenager(x)) & WishToFurther(x, academicCareer)) -> Studen(x) & AttendSchool(x)
                                         ^
Example 0 parsing error:
InThisClub(bonnie) ∧ ¬((Attend(x, schoolEvent) ∧ VeryEngagedWith(bonnie, schoolEvent)) ⊕ (Studen(bonne) ∧ AttendSchool(bonnie)))
Unexpected token: '⊕'.  Expected token ')'.
InThisClub(bonnie) & -((Attend(x, schoolEvent) & VeryEngagedWith(bonnie, schoo

## FOLIO implementation (new)

This cell adds a recursive Unicode-FOL converter for NLTK. It expands exclusive OR (`⊕`) as `(A OR B) AND NOT (A AND B)`, handles nested expressions, and removes surplus closing parentheses observed in some supplied rows. The evaluation reports **coverage and errors** as well as accuracy; it does not silently count failed examples as predictions.

Run the parser check first, then run the validation evaluation cell. The existing CounterBench solver remains a preliminary baseline and should not be described as a completed causal reasoner.

In [ ]:
import re
import time
from nltk.sem.logic import LogicParser
from nltk.inference import ResolutionProver

folio_lp = LogicParser()

def _strip_full_outer_parentheses(s):
    s = s.strip()
    while s.startswith("(") and s.endswith(")"):
        depth = 0
        wraps_all = True
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
                if depth == 0 and i != len(s) - 1:
                    wraps_all = False
                    break
        if wraps_all:
            s = s[1:-1].strip()
        else:
            break
    return s

def _split_top_level(s, operator):
    depth, positions, i = 0, [], 0
    while i < len(s):
        if s[i] == "(":
            depth += 1
        elif s[i] == ")":
            depth -= 1
        elif depth == 0 and s.startswith(operator, i):
            positions.append(i)
            i += len(operator)
            continue
        i += 1
    if not positions:
        return None
    parts, last = [], 0
    for pos in positions:
        parts.append(s[last:pos].strip())
        last = pos + len(operator)
    parts.append(s[last:].strip())
    return parts

def convert_folio_formula(formula):
    s = formula.strip()
    s = s.replace("∧", "&").replace("∨", "|").replace("¬", "-")
    s = s.replace("→", "->")

    # Remove surplus closing parentheses without altering balanced groups.
    depth, balanced = 0, ""
    for ch in s:
        if ch == "(":
            depth += 1
            balanced += ch
        elif ch == ")":
            if depth > 0:
                depth -= 1
                balanced += ch
        else:
            balanced += ch
    s = _strip_full_outer_parentheses(balanced)

    # Quantifiers bind over the remaining expression.
    qm = re.match(r"^([∀∃])\s*([A-Za-z_]\w*)\s*(.*)$", s)
    if qm:
        quantifier, variable, body = qm.groups()
        q = "all" if quantifier == "∀" else "exists"
        return f"{q} {variable}.({convert_folio_formula(body)})"
    qm = re.match(r"^(all|exists)\s+([A-Za-z_]\w*)\s*(.*)$", s)
    if qm:
        q, variable, body = qm.groups()
        return f"{q} {variable}.({convert_folio_formula(body)})"

    # Operator precedence: implication, XOR, OR, AND.
    for op in ("->", "⊕", "|", "&"):
        parts = _split_top_level(s, op)
        if parts:
            converted = [convert_folio_formula(part) for part in parts]
            if op == "->":
                result = converted[-1]
                for left in reversed(converted[:-1]):
                    result = f"({left} -> {result})"
                return result
            if op == "⊕":
                result = converted[0]
                for right in converted[1:]:
                    result = f"(({result} | {right}) & -({result} & {right}))"
                return result
            return "(" + f" {op} ".join(converted) + ")"

    if s.startswith("-"):
        return "-(" + convert_folio_formula(s[1:]) + ")"
    return s

def parse_folio_formula(formula):
    return folio_lp.parse(convert_folio_formula(formula))

def predict_folio_row(row):
    premises = [parse_folio_formula(x) for x in row["premises-FOL"].splitlines() if x.strip()]
    conclusion = parse_folio_formula(row["conclusion-FOL"])
    positive = ResolutionProver().prove(conclusion, premises)
    negative = ResolutionProver().prove(-conclusion, premises)
    if positive and not negative:
        return "True"
    if negative and not positive:
        return "False"
    return "Uncertain"

print("FOLIO parser and predictor loaded.")

In [ ]:
# First check parsing on five validation examples.
parsed, failed = 0, 0
for i in range(min(5, len(folio["validation"]))):
    row = folio["validation"][i]
    for formula in row["premises-FOL"].splitlines() + [row["conclusion-FOL"]]:
        if not formula.strip():
            continue
        try:
            parse_folio_formula(formula)
            parsed += 1
        except Exception as exc:
            failed += 1
            print("Parse error:", i, str(exc)[:250])
print("Parsed:", parsed, "Failed:", failed)

# Evaluate only after the parser check. Failures are excluded from accuracy
# but explicitly counted in coverage.
start = time.perf_counter()
correct = evaluated = errors = 0
first_errors = []
for row in folio["validation"]:
    try:
        prediction = predict_folio_row(row)
        evaluated += 1
        correct += int(prediction == row["label"])
    except Exception as exc:
        errors += 1
        if len(first_errors) < 5:
            first_errors.append((row.get("example_id"), str(exc)[:250]))
elapsed = time.perf_counter() - start
print("Total:", len(folio["validation"]))
print("Evaluated:", evaluated)
print("Errors:", errors)
print("Coverage (%):", round(100 * evaluated / len(folio["validation"]), 2))
print("Accuracy on evaluated (%):", round(100 * correct / evaluated, 2) if evaluated else "N/A")
print("Runtime (seconds):", round(elapsed, 3))
print("First errors:", first_errors)